# HonestDiD: анализ чувствительности

## Назначение
Кратко: анализ чувствительности по Rambachan & Roth (2023) для основных DiD-оценок last-mile CORE при ограниченных нарушениях предположения о параллельных трендах. Ноутбук использует гетерогенно-устойчивые оценки событийная модель Callaway & Sant'Anna (`differences.ATTgt`) и вызывает **официальный R-пакет `HonestDiD`** через `Rscript`; границы не реализуются в Python и TWFE не используется как основной вход.

> **Предупреждение об области применения — что HonestDiD не исправляет.** Метод ослабляет только предположение о параллельных трендах. Он не учитывает anticipation/selection, правую цензуру медленных исходов, attrition, ошибки измерения, SUTVA/спillover или слабую идентификацию. Статистически неоднозначная базовая оценка остаётся неоднозначной при любых параметрах чувствительности. Дозовые веса $W_h^+$/$W_h^-$ не связаны с параметрами HonestDiD $M$/$\bar{M}$.

## Входные данные
- канонические коэффициенты событийной модели / ATT из предшествующих расчётов (`differences.ATTgt`);
- спецификации исходов CORE и реестр когорт;
- при наличии — веса пост-периода из `outputs/final/event_study_post_weights.csv`.

## Результаты
- таблицы и графики в `outputs/honest_did/` и `figures/honest_did/`;
- сводный CSV (`honest_did_summary.csv`, реестр моделей, порог устойчивости).

## Статус
Диагностика (анализ чувствительности), не основная спецификация.

**Воспроизводимость:** фиксированный seed `20260712`, пути через `pathlib`, без изменения исходных фреймов на месте. Полный *полный перезапуск и выполнение всех ячеек* требует пакет `differences` и R с `HonestDiD`, `jsonlite`, `readr`. Раздел 10 завершается с инструкцией по установке, если R-сторона не готова.

## 1. Среда и воспроизводимость (проверка R)

Фиксируется seed и определяется `Rscript` только через `PATH`. Готовность
R-пакетов проверяется перед мостом HonestDiD; ранние ячейки с данными и
диагностикой не зависят от локальной конфигурации R.


In [ ]:
from __future__ import annotations

import itertools
import json
import locale
import os
import random
import shutil
import subprocess
import sys
import time
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm

SEED = 20260712
random.seed(SEED)
np.random.seed(SEED)

RUN_TIMESTAMP = datetime.now(timezone.utc).isoformat()


def decode_subprocess_output(value) -> str:
    """Декодирует вывод подпроцесса без утечки UnicodeDecodeError."""
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    if not isinstance(value, (bytes, bytearray, memoryview)):
        return str(value)
    raw = bytes(value)
    encodings = ["utf-8", locale.getpreferredencoding(False), "cp1251"]
    seen = set()
    for encoding in encodings:
        key = (encoding or "").lower()
        if not key or key in seen:
            continue
        seen.add(key)
        try:
            return raw.decode(encoding)
        except (UnicodeDecodeError, LookupError):
            pass
    return raw.decode("utf-8", errors="replace")


@contextmanager
def cell_progress(description: str, total: int = 1, unit: str = "step"):
    """Показывает прогресс ячейки и не отмечает незавершённую работу как успешную."""
    bar = tqdm(
        total=int(total),
        desc=description,
        unit=unit,
        leave=True,
        dynamic_ncols=True,
        mininterval=0.5,
    )
    try:
        yield bar
    except BaseException:
        bar.set_description_str(f"{description} [ошибка]")
        bar.set_postfix(status="ошибка", refresh=True)
        raise
    else:
        remaining = max(int(bar.total or 0) - int(bar.n), 0)
        if remaining:
            bar.update(remaining)
        bar.set_description_str(f"{description} [завершено]")
        bar.set_postfix(status="завершено", refresh=True)
    finally:
        bar.close()


def find_rscript() -> str | None:
    """Возвращает Rscript из PATH; локальные пути не используются."""
    return shutil.which("Rscript")


with cell_progress("Подготовка окружения и версия R", total=1) as progress:
    RSCRIPT_PATH = find_rscript()
    R_VERSION = None
    if RSCRIPT_PATH is not None:
        proc = subprocess.run(
            [RSCRIPT_PATH, "-e", "cat(as.character(getRversion()))"],
            capture_output=True,
            text=False,
        )
        stdout = decode_subprocess_output(proc.stdout)
        _ = decode_subprocess_output(proc.stderr)
        if proc.returncode == 0:
            R_VERSION = stdout.strip()

    print("Версия Python:", sys.version.split()[0])
    print("Начальное значение:", SEED, "| время запуска (UTC):", RUN_TIMESTAMP)
    print("Rscript:", RSCRIPT_PATH if RSCRIPT_PATH else "НЕ НАЙДЕН")
    print("Версия R:", R_VERSION if R_VERSION else "неизвестна")


## 2. Импорты и константы

Бизнес-логика импортируется из `last_mile`; здесь ничего не выводится заново. Тонкие обёртки ниже повторяют построители выборок из `final_empirical_recalculation.ipynb` (`prepare_utlz_orders`, `cohorts_for_outcome`, фильтры зрелости, `make_treated/control`, `prepare_differences_panel`, `run_differences_attgt` и shim кластеризации для `differences==0.3.0`). Функцию `classify_hexagons` **не** переопределяем.

In [18]:
with cell_progress("Загрузка конфигурации", total=1) as progress:
    import importlib
    import importlib.util
    import inspect
    import warnings

    import pandas as pd
    import matplotlib.pyplot as plt

    # --- Корень проекта и настройка путей -------------------------------------------------
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd.parent if cwd.name.lower() in {"notebooks", "notebook"} else cwd
    if not (PROJECT_ROOT / "last_mile").exists() and (cwd / "last_mile").exists():
        PROJECT_ROOT = cwd
    for candidate in [PROJECT_ROOT, PROJECT_ROOT / "last_mile"]:
        if candidate.exists() and str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))

    DATA_RAW = PROJECT_ROOT / "data" / "raw"
    APPLICATIONS_PATH = DATA_RAW / "application_dataset.csv"
    HEXAGONS_PATH = DATA_RAW / "hexagons_dataset.csv"

    OUT_DIR = PROJECT_ROOT / "outputs" / "honest_did"
    FIG_DIR = PROJECT_ROOT / "figures" / "honest_did"
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    FINAL_DIR = PROJECT_ROOT / "outputs" / "final"

    # --- Бизнес-логика из last_mile ---------------------------------------------
    from last_mile import (
        build_analysis_panel,
        build_did_samples,
        CORE_CHANGE_TYPES,
    )
    from last_mile.filter import EXCLUDED_COHORTS
    from last_mile.cohort_diagnostics import EXCLUDED_COHORT_FOR_CONVERSION
    from last_mile.outcomes import success_horizon_coverage
    from last_mile.plot_style import plot_style, save_figure, PALETTE

    # --- Константы -----------------------------------------------------------------
    CORE_ONLY = list(CORE_CHANGE_TYPES)  # region_only, workmode_only, region_and_workmode
    POOLED_LABEL = "CORE_pooled"
    CHANGE_TYPES = CORE_ONLY + [POOLED_LABEL]

    # Когорта 2022-07-27 исключена для конверсий и сохранена для sch/t_available.
    COHORT_0727 = pd.Timestamp(EXCLUDED_COHORT_FOR_CONVERSION)
    CONVERSION_OUTCOMES_EXCLUDE_0727 = {
        "meet_flg", "success_within_20", "utlz_within_25", "utlz_flg"
    }

    SPEED_OUTCOME = "t_available"
    UTLZ_OUTCOME = "utlz_within_25"
    UTLZ_OUTCOMES = {"utlz_within_25", "utlz_flg"}
    UTLZ_HORIZON_DAYS = 25
    UTILIZATION_OBSERVATION_END = pd.Timestamp("2022-10-18")
    TIME_OUTCOMES = {"t_available", "t_utilization"}

    # Доли оцениваются как дроби и только на графиках переводятся в процентные пункты.
    PRIMARY_OUTCOMES = [
        "sch_flg", "meet_flg", "success_within_20", "utlz_within_25"
    ]
    OPTIONAL_OUTCOMES = ["t_available"]        # добавляется при успешном ATTgt с полной ковариационной матрицей
    EXTRA_OUTCOMES = ["utlz_flg"]              # дополнительный исход с неограниченным правым цензурированием

    # --- Параметры запуска -----------------------------------------------------------------
    RUN_ATTGT = True                 # пересчитывать входы ATTgt; иначе использовать проверенные betahat/sigma из кэша
    FORCE_CLUSTER_BOOTSTRAP = False  # предпочтительны функции влияния; bootstrap используется только при их отсутствии
    N_BOOTSTRAP = 999                # при резервном bootstrap предпочтительно не менее 499 повторов
    EVENT_WEEK_MIN = -8
    EVENT_WEEK_MAX = 8
    REFERENCE_WEEK = -1
    ALPHA = 0.05

    # --- Готовность HonestDiD (заполняется в разделе 10) -----------------------------
    HONESTDID_READY = False
    R_PACKAGE_STATUS: dict[str, bool] = {}

    # --- Реестры, заполняемые в ноутбуке ---------------------------------
    failed_models: list[dict] = []
    model_registry_rows: list[dict] = []

    CONCLUSION_CODES = {
        "robust_at_reported_range",
        "sign_sensitive_to_moderate_violations",
        "baseline_inconclusive",
        "insufficient_support",
        "computation_failed",
    }

    print("PROJECT_ROOT:", PROJECT_ROOT)
    print("OUT_DIR:", OUT_DIR)
    print("FIG_DIR:", FIG_DIR)
    print("CORE change types:", CORE_ONLY, "| pooled:", POOLED_LABEL)
    print("Excluded cohorts (right-censoring):", sorted(str(c.date()) for c in EXCLUDED_COHORTS))
    print("Cohort excluded for conversion outcomes:", COHORT_0727.date())

PROJECT_ROOT: <resolved relative to repository root>
OUT_DIR: <under repository root>
FIG_DIR: <under repository root>
CORE change types: ['region_only', 'workmode_only', 'region_and_workmode'] | pooled: CORE_pooled
Excluded cohorts (right-censoring): ['2022-10-19']
Cohort excluded for conversion outcomes: 2022-07-27


## 3. Загрузка существующего пайплайна и результатов проекта

Строим панель гексагон-день и выборки на уровне заявок через `build_analysis_panel` / `build_did_samples`, затем определяем тонкие обёртки построения выборок по образцу `final_empirical_recalculation.ipynb`. Обёртки не мутируют исходные фреймы на месте и не удаляют дубликаты по `hex` молча.

In [19]:
with cell_progress("Build analysis panel", total=1) as progress:
    # --- Панель гексагон-день и таблицы заявок из того же источника, что и в финальном расчёте
    # build_analysis_panel применяет compute_time_variables и добавляет t_available.
    panel_result = build_analysis_panel(APPLICATIONS_PATH, HEXAGONS_PATH, min_orders_per_hex=5)
    panel = panel_result["panel"].copy()
    apps_t = panel_result["treated_orders"].copy()
    apps_c = panel_result["control_orders"].copy()

    for frame in (apps_t, apps_c):
        if "cohort" in frame.columns and "treatment_date" not in frame.columns:
            frame["treatment_date"] = frame["cohort"]
        if "date" not in frame.columns:
            frame["date"] = frame["request_timestamp"].dt.normalize()
    if "days_from_treatment" not in apps_t.columns:
        apps_t["days_from_treatment"] = (
            apps_t["request_timestamp"] - apps_t["treatment_date"]
        ).dt.days

    # Временной исход должен присутствовать для дополнительного расчёта ATTgt
    print("t_available in apps_t:", "t_available" in apps_t.columns,
          "| in panel:", "t_available" in panel.columns)

    OBS_MIN = min(apps_t["request_timestamp"].min(), apps_c["request_timestamp"].min())
    OBS_MAX = max(apps_t["request_timestamp"].max(), apps_c["request_timestamp"].max())


    def _neg_utilization_lag_mask(df: pd.DataFrame) -> pd.Series:
        """Маска заявок, в которых дата утилизации предшествует дате заявки."""
        return (
            df["real_utilization_dttm"].notna()
            & (df["real_utilization_dttm"] < df["request_timestamp"])
        )


    def prepare_utlz_orders(df: pd.DataFrame, horizon_days: int = UTLZ_HORIZON_DAYS) -> pd.DataFrame:
        """Удаляет отрицательные лаги и строит ограниченный исход ``utlz_within_H``.

        Повторяет ``final_empirical_recalculation.prepare_utlz_orders`` и возвращает
        копию без изменения исходного фрейма.
        """
        out = df.loc[~_neg_utilization_lag_mask(df)].copy()
        horizon = out["request_timestamp"] + pd.Timedelta(days=horizon_days)
        out[UTLZ_OUTCOME] = (
            out["real_utilization_dttm"].notna() & (out["real_utilization_dttm"] <= horizon)
        ).astype(float)
        return out


    apps_t_utlz = prepare_utlz_orders(apps_t)
    apps_c_utlz = prepare_utlz_orders(apps_c)
    apps_t, _ = success_horizon_coverage(
        apps_t, horizons=(20,), observation_end=UTILIZATION_OBSERVATION_END
    )
    apps_c, _ = success_horizon_coverage(
        apps_c, horizons=(20,), observation_end=UTILIZATION_OBSERVATION_END
    )

    # Допустимые когорты: все CORE-когорты; для конверсий исключается 2022-07-27.
    VALID_COHORTS_ALL = {pd.Timestamp(x) for x in apps_t["treatment_date"].dropna().unique()}
    VALID_COHORTS_CONVERSION = VALID_COHORTS_ALL - {COHORT_0727}


    def cohorts_for_outcome(outcome: str) -> set[pd.Timestamp]:
        """Возвращает допустимые когорты воздействия для заданного исхода."""
        if outcome in CONVERSION_OUTCOMES_EXCLUDE_0727:
            return set(VALID_COHORTS_CONVERSION)
        return set(VALID_COHORTS_ALL)


    def outcome_agg_func(outcome: str) -> str:
        """Агрегация гексагон-день: медиана длительностей и среднее бинарных флагов."""
        return "median" if outcome in TIME_OUTCOMES else "mean"


    OUTCOME_EVENT_DTTM = {
        "utlz_flg": "real_utilization_dttm",
    }


    def maturity_horizon_days(outcome: str, quantile: float = 0.90) -> int:
        """Горизонт зрелости H в днях до наблюдения исхода.

        Для утилизации используется 25 дней, для sch/meet/t_available — 0,
        для остальных исходов — p90 лага от заявки до события.
        """
        if outcome == "success_within_20":
            return 20
        if outcome in UTLZ_OUTCOMES:
            return int(UTLZ_HORIZON_DAYS)
        event_col = OUTCOME_EVENT_DTTM.get(outcome)
        if event_col is None or event_col not in apps_t.columns or event_col not in apps_c.columns:
            return 0
        lag_df = pd.concat(
            [apps_t[["request_timestamp", event_col]], apps_c[["request_timestamp", event_col]]],
            ignore_index=True,
        ).dropna(subset=["request_timestamp", event_col])
        if lag_df.empty:
            return 0
        lags = (lag_df[event_col] - lag_df["request_timestamp"]).dt.total_seconds() / 86400.0
        lags = lags[(lags >= 0) & np.isfinite(lags)]
        return int(np.ceil(lags.quantile(quantile))) if not lags.empty else 0


    def filter_mature_requests(df: pd.DataFrame, outcome: str, quantile: float = 0.90):
        """Оставляет заявки с достаточным периодом наблюдения и возвращает копию."""
        h = maturity_horizon_days(outcome, quantile=quantile)
        if h <= 0:
            return df.copy(), h
        obs_end = (
            UTILIZATION_OBSERVATION_END
            if outcome in UTLZ_OUTCOMES or outcome == "success_within_20"
            else OBS_MAX
        )
        return df[df["request_timestamp"] + pd.Timedelta(days=h) <= obs_end].copy(), h


    def _treated_source(outcome: str) -> pd.DataFrame:
        return apps_t_utlz if outcome in UTLZ_OUTCOMES else apps_t


    def _control_source(outcome: str) -> pd.DataFrame:
        return apps_c_utlz if outcome in UTLZ_OUTCOMES else apps_c


    def make_treated_sample(outcome: str, change_type: str) -> pd.DataFrame:
        """Заявки воздействия одного CORE-типа или объединённой CORE-выборки после фильтра зрелости."""
        cohorts = cohorts_for_outcome(outcome)
        src = _treated_source(outcome)
        if change_type == POOLED_LABEL:
            type_mask = src["change_type"].isin(CORE_CHANGE_TYPES)
        else:
            type_mask = src["change_type"] == change_type
        treated = src[type_mask & src["treatment_date"].isin(cohorts)].copy()
        treated, _ = filter_mature_requests(treated, outcome)
        return treated


    def make_control_sample(outcome: str) -> pd.DataFrame:
        """Заявки never-treated контроля после фильтра зрелости; возвращается копия."""
        control, _ = filter_mature_requests(_control_source(outcome).copy(), outcome)
        return control


    # --- Веса постпериода для заявок воздействия из подтверждённого финального расчёта ------------------
    POST_WEIGHTS_PATH = FINAL_DIR / "event_study_post_weights.csv"
    post_weights_df = None
    if POST_WEIGHTS_PATH.exists():
        post_weights_df = pd.read_csv(POST_WEIGHTS_PATH)
        print("Загружены веса постпериода для заявок воздействия:", POST_WEIGHTS_PATH.name,
              "| rows:", len(post_weights_df))
    else:
        print("[WARN] event_study_post_weights.csv not found; main_post_average will fall back "
              "to treated-order counts computed on the fly.")

    print("заявок воздействия:", len(apps_t), "| контрольных заявок:", len(apps_c))
    print("VALID_COHORTS_ALL:", sorted(str(c.date()) for c in VALID_COHORTS_ALL))
    print("VALID_COHORTS_CONVERSION:", sorted(str(c.date()) for c in VALID_COHORTS_CONVERSION))


[INFO] resolve_hexagon_treatments: исключено 15832 multi-treatment гексагонов; сохранено 3613240 single/never.
[INFO] multi-treatment diagnostics: <PROJECT_ROOT>/…
[INFO] attach_hex_metadata: отброшено 109 заявок из гексагонов, не представленных в hexagons_dataset.
[INFO] exclude_cohorts: исключено 683082 заявок из когорт ['2022-10-19'].
[INFO] build_analysis_panel: после порогового фильтра (≥5 заявок) осталось 13255 уникальных гексагонов.

СВОДКА АНАЛИТИЧЕСКОЙ ПАНЕЛИ
  Период:              2022-04-01 - 2022-10-18
  Наблюдений (hex×day):   302,398
  Гексагонов всего:        13,255
  - трактуемых:             9,327
  - never-treated:          3,928

  Когорты (включённые):
    2022-07-27  →  437 гексагонов
    2022-08-12  →  1358 гексагонов
    2022-08-19  →  2723 гексагонов
    2022-09-22  →  4809 гексагонов

  Исключённые когорты: ['2022-10-19']

t_available in apps_t: True | in panel: True
Загружены веса постпериода для заявок воздействия: event_study_post_weights.csv | rows: 117
зая

## 4. Реестр спецификаций исходов и когорт

По одной строке на исход: правило выборки, исключения когорт (с причиной), единица измерения и tier (primary / optional / extra с правой цензурой). Когорта 2022-07-27 показана явно с причиной включения/исключения.

In [20]:
with cell_progress("Register outcomes", total=1) as progress:
    OUTCOME_SPECS = [
        {
            "outcome": "sch_flg",
            "tier": "primary",
            "unit": "proportion",
            "cohort_0727": "included",
            "cohort_0727_reason": "scheduling outcome valid for 2022-07-27; no post-meeting pretrend issue",
            "sample_rule": "CORE treated vs never-treated; all valid cohorts; maturity H=0",
            "note": "",
        },
        {
            "outcome": "meet_flg",
            "tier": "primary",
            "unit": "proportion",
            "cohort_0727": "excluded",
            "cohort_0727_reason": "parallel-trends violation for post-meeting outcomes (pretrend p ~ 0.0168)",
            "sample_rule": "CORE treated vs never-treated; conversion cohorts; maturity H=0",
            "note": "",
        },
        {
            "outcome": "success_within_20",
            "tier": "primary",
            "unit": "proportion",
            "cohort_0727": "excluded",
            "cohort_0727_reason": "post-meeting conversion outcome; 2022-07-27 excluded with meet_flg",
            "sample_rule": "CORE treated vs never-treated; conversion cohorts; fixed maturity H=20d",
            "note": "event is first_success_dttm within 20 days; lifetime success_flg is not used",
        },
        {
            "outcome": "utlz_within_25",
            "tier": "primary",
            "unit": "proportion",
            "cohort_0727": "excluded",
            "cohort_0727_reason": "capped utilization is a post-meeting conversion outcome; excluded",
            "sample_rule": "CORE treated vs never-treated; conversion cohorts; H=25d; neg lags dropped",
            "note": "capped at 25 days to avoid right-censoring",
        },
        {
            "outcome": "t_available",
            "tier": "optional",
            "unit": "days",
            "cohort_0727": "included",
            "cohort_0727_reason": "availability latency is a pre-meeting outcome; 2022-07-27 retained",
            "sample_rule": "CORE treated vs never-treated; all valid cohorts; median hex-day agg",
            "note": "included only if ATTgt with full covariance succeeds; known pretrend caveats",
        },
        {
            "outcome": "utlz_flg",
            "tier": "extra",
            "unit": "proportion",
            "cohort_0727": "excluded",
            "cohort_0727_reason": "post-meeting conversion outcome; excluded with the other conversion flags",
            "sample_rule": "CORE treated vs never-treated; conversion cohorts; UNCAPPED utilization",
            "note": "RIGHT-CENSORING ISSUE: uncapped utilization mixes short and unbounded horizons",
        },
    ]

    outcome_spec_table = pd.DataFrame(OUTCOME_SPECS)
    # Исходы, входящие в оценивание; дополнительный неограниченный исход включается через ENABLE_EXTRA.
    ENABLE_EXTRA = False
    ANALYSIS_OUTCOMES = list(PRIMARY_OUTCOMES) + list(OPTIONAL_OUTCOMES)
    if ENABLE_EXTRA:
        ANALYSIS_OUTCOMES = ANALYSIS_OUTCOMES + list(EXTRA_OUTCOMES)

    outcome_spec_table.to_csv(OUT_DIR / "outcome_specification_registry.csv", index=False)
    print("Outcomes entering estimation:", ANALYSIS_OUTCOMES)
    print("Extra (uncapped) enabled:", ENABLE_EXTRA)
    try:
        from IPython.display import display
        display(outcome_spec_table)
    except Exception:
        print(outcome_spec_table.to_string(index=False))

Outcomes entering estimation: ['sch_flg', 'meet_flg', 'success_flg', 'utlz_within_25', 't_available']
Extra (uncapped) enabled: False


          outcome      tier        unit cohort_0727  \
0         sch_flg   primary  proportion    included   
1        meet_flg   primary  proportion    excluded   
2     success_flg   primary  proportion    excluded   
3  utlz_within_25   primary  proportion    excluded   
4     t_available  optional        days    included   
5        utlz_flg     extra  proportion    excluded   

                                  cohort_0727_reason  \
0  scheduling outcome valid for 2022-07-27; no po...   
1  parallel-trends violation for post-meeting out...   
2  post-meeting conversion outcome; 2022-07-27 ex...   
3  capped utilization is a post-meeting conversio...   
4  availability latency is a pre-meeting outcome;...   
5  post-meeting conversion outcome; excluded with...   

                                         sample_rule  \
0  CORE treated vs never-treated; all valid cohor...   
1  CORE treated vs never-treated; conversion coho...   
2  CORE treated vs never-treated; conversion coho... 

## 5. Диагностика сбалансированной поддержки

Для каждой пары `outcome × change_type` до оценивания выбирается **сбалансированное окно событийной недели**. Предпочтение: `-8 … +8` без референсной недели `-1`, не менее двух предпериодных коэффициентов, референс `-1` (нормирован в 0, исключён из `betahat`) и не менее одного постпериодного коэффициента. Окно обрезается до наибольшего непрерывного набора событийных недель с **общим набором когорт** (когорты с заявками воздействия и на ключевой преднеделе `-2`, и на неделе воздействия `0`), чтобы одни и те же когорты участвовали на каждом event time. Если условие не выполняется, модель помечается `insufficient_support` с причиной и пропускается при оценивании.

In [21]:
with cell_progress("Check balanced support", total=len(ANALYSIS_OUTCOMES) * len(CHANGE_TYPES), unit="model") as progress:
    def _treated_with_relweek(outcome: str, change_type: str) -> pd.DataFrame:
        treated = make_treated_sample(outcome, change_type)
        if treated.empty:
            return treated
        treated = treated.copy()
        # Обрезка применяется только в таблицах поддержки; HonestDiD использует непрерывные недели без группировки.
        treated["rel_week"] = (treated["days_from_treatment"] // 7).clip(EVENT_WEEK_MIN, EVENT_WEEK_MAX)
        treated["cohort"] = pd.to_datetime(treated["treatment_date"])
        return treated


    def choose_balanced_event_window(outcome: str, change_type: str) -> dict:
        """Наибольшее непрерывное окно событийных недель с одинаковым набором когорт.

        Требования: не менее двух коэффициентов предпериода без референса -1; неделя -1 входит
        в интервал поддержки; есть хотя бы один коэффициент постпериода; состав когорт постоянен.
        """
        treated = _treated_with_relweek(outcome, change_type)
        base: dict = {
            "outcome": outcome,
            "change_type": change_type,
            "reference_week": REFERENCE_WEEK,
            "betahat_weeks": [],
            "pre_weeks": [],
            "post_weeks": [],
            "window_weeks": [],
            "allowed_cohorts": [],
            "n_common_cohorts": 0,
            "n_hex": 0,
            "n_orders": 0,
            "support_table": pd.DataFrame(),
            "support_long": pd.DataFrame(),
        }
        if treated.empty:
            base.update(status="insufficient_support", reason="empty treated sample")
            return base

        treated = treated.dropna(subset=["rel_week"]).copy()
        treated["rel_week"] = treated["rel_week"].astype(int)

        support_long = (
            treated.groupby(["cohort", "rel_week"], as_index=False)
            .agg(n_hex=("hex", "nunique"), n_orders=("request_timestamp", "count"))
        )
        support_table = support_long.pivot_table(
            index="cohort", columns="rel_week", values="n_hex", fill_value=0
        ).astype(int)

        def cohorts_with_week(week: int) -> set:
            if week not in support_table.columns:
                return set()
            return set(support_table.index[support_table[week] > 0])

        candidates: list[tuple[int, int]] = []
        for lo in range(EVENT_WEEK_MIN, 0):
            for hi in range(0, EVENT_WEEK_MAX + 1):
                weeks = list(range(lo, hi + 1))
                if REFERENCE_WEEK not in weeks:
                    continue
                pre_omitted = [w for w in weeks if w < 0 and w != REFERENCE_WEEK]
                post = [w for w in weeks if w >= 0]
                if len(pre_omitted) < 2 or len(post) < 1:
                    continue
                candidates.append((lo, hi))
        candidates.sort(
            key=lambda x: (-(x[1] - x[0]), abs(x[0] - EVENT_WEEK_MIN) + abs(x[1] - EVENT_WEEK_MAX))
        )

        best = None
        for lo, hi in candidates:
            weeks = list(range(lo, hi + 1))
            sets = [cohorts_with_week(w) for w in weeks]
            if any(len(s) == 0 for s in sets):
                continue
            common = set.intersection(*sets)
            if len(common) < 2:
                continue
            pre_omitted = [w for w in weeks if w < 0 and w != REFERENCE_WEEK]
            post = [w for w in weeks if w >= 0]
            sub = treated[treated["cohort"].isin(common) & treated["rel_week"].isin(weeks)]
            best = {
                "status": "ok",
                "reason": "balanced continuous support with fixed cohort composition",
                "window_lo": lo,
                "window_hi": hi,
                "window_weeks": weeks,
                "betahat_weeks": pre_omitted + post,
                "pre_weeks": pre_omitted,
                "post_weeks": post,
                "allowed_cohorts": [str(pd.Timestamp(c).date()) for c in sorted(common)],
                "n_common_cohorts": len(common),
                "n_hex": int(sub["hex"].nunique()),
                "n_orders": int(len(sub)),
                "support_table": support_table,
                "support_long": support_long,
            }
            break

        if best is None:
            base.update(
                status="insufficient_support",
                reason=(
                    "could not find a continuous window with identical cohorts on every event week, "
                    f">=2 pre (excl. ref {REFERENCE_WEEK}), and >=1 post"
                ),
                support_table=support_table,
                support_long=support_long,
            )
            return base

        base.update(best)
        return base


    balanced_windows: dict[tuple[str, str], dict] = {}
    balanced_rows = []
    support_diag_rows = []
    for _outcome, _ctype in itertools.product(ANALYSIS_OUTCOMES, CHANGE_TYPES):
        progress.set_postfix(outcome=_outcome, change_type=_ctype, refresh=False)
        info = choose_balanced_event_window(_outcome, _ctype)
        balanced_windows[(_outcome, _ctype)] = info
        balanced_rows.append({
            "outcome": _outcome,
            "change_type": _ctype,
            "status": info["status"],
            "n_pre": len(info["pre_weeks"]),
            "n_post": len(info["post_weeks"]),
            "pre_weeks": ",".join(map(str, info["pre_weeks"])),
            "post_weeks": ",".join(map(str, info["post_weeks"])),
            "balanced_event_window": (
                f"[{info.get('window_lo', '')},{info.get('window_hi', '')}] omit {REFERENCE_WEEK}"
                if info["status"] == "ok"
                else ""
            ),
            "n_common_cohorts": info["n_common_cohorts"],
            "n_hex": info.get("n_hex", 0),
            "n_orders": info.get("n_orders", 0),
            "allowed_cohorts": "; ".join(info["allowed_cohorts"]),
            "reason": info["reason"],
        })
        if info["status"] == "ok" and not info["support_long"].empty:
            common = set(pd.to_datetime(info["allowed_cohorts"]))
            for w in info["window_weeks"]:
                sub = info["support_long"][
                    (info["support_long"]["rel_week"] == w)
                    & (info["support_long"]["cohort"].isin(common))
                ]
                support_diag_rows.append({
                    "outcome": _outcome,
                    "change_type": _ctype,
                    "event_time": w,
                    "n_cohorts": int(sub["cohort"].nunique()),
                    "n_hex": int(sub["n_hex"].sum()),
                    "n_orders": int(sub["n_orders"].sum()),
                    "included_cohorts": "; ".join(info["allowed_cohorts"]),
                })
        progress.update(1)
    balanced_support_summary = pd.DataFrame(balanced_rows)
    balanced_support_summary.to_csv(OUT_DIR / "balanced_support_diagnostics.csv", index=False)
    support_diagnostics = pd.DataFrame(support_diag_rows)
    support_diagnostics.to_csv(OUT_DIR / "support_diagnostics_long.csv", index=False)
    display(balanced_support_summary)
    print("Models marked insufficient_support:")
    display(balanced_support_summary.loc[balanced_support_summary["status"] != "ok"])


           outcome          change_type status  n_pre  n_post  \
0          sch_flg          region_only     ok      7       4   
1          sch_flg        workmode_only     ok      7       9   
2          sch_flg  region_and_workmode     ok      7       9   
3          sch_flg          CORE_pooled     ok      7       9   
4         meet_flg          region_only     ok      7       4   
5         meet_flg        workmode_only     ok      7       9   
6         meet_flg  region_and_workmode     ok      7       9   
7         meet_flg          CORE_pooled     ok      7       9   
8      success_flg          region_only     ok      7       1   
9      success_flg        workmode_only     ok      7       6   
10     success_flg  region_and_workmode     ok      7       6   
11     success_flg          CORE_pooled     ok      7       6   
12  utlz_within_25          region_only     ok      7       1   
13  utlz_within_25        workmode_only     ok      7       6   
14  utlz_within_25  regio

Models marked insufficient_support:


,outcome,change_type,status,n_pre,n_post,pre_weeks,post_weeks,balanced_event_window,n_common_cohorts,n_hex,n_orders,allowed_cohorts,reason


## 6. Получение гетерогенно-устойчивых оценок событийная модель

Основной оценщик: **`differences.ATTgt`** (Callaway & Sant'Anna) с `control_group="never_treated"`, `base_period="universal"`, `anticipation=0`, `freq="D"` и кластеризацией по `cluster_hex`. Сохраняем fitted object и namedtuple event-агрегации, чтобы в разделе 7 извлечь influence functions (`to_pandas` их отбрасывает). Оценивание управляется флагом `RUN_ATTGT`; ATTgt медленный, полный перезапуск и выполнение всех ячеек занимает много времени и дополнительно требует R для разделов 10+. При `RUN_ATTGT=False` можно переиспользовать кэшированные `betahat`/`sigma` из `outputs/honest_did`, если они есть.

In [22]:
with cell_progress("Fit ATTgt models", total=(sum(info["status"] == "ok" for info in balanced_windows.values()) if RUN_ATTGT else 0), unit="model") as progress:
    def _patch_differences_cluster_compat():
        """Для differences==0.3.0 аргумент кластеров Series преобразуется в DataFrame."""
        import differences.models.attgt.attgt as _attgt_mod
        import differences.models.attgt.mboot as _mboot
        import differences.models.attgt.results as _results_mod

        if getattr(_patch_differences_cluster_compat, "_applied", False):
            return
        _original = _mboot.get_cluster_groups

        def _get_cluster_groups(data, cluster_var=None):
            if isinstance(data, pd.Series):
                col = data.name
                if col is None:
                    if isinstance(cluster_var, list) and cluster_var:
                        col = cluster_var[0]
                    elif isinstance(cluster_var, str):
                        col = cluster_var
                    else:
                        col = "cluster_var"
                data = data.to_frame(name=col)
            return _original(data, cluster_var)

        for mod in (_mboot, _attgt_mod, _results_mod):
            mod.get_cluster_groups = _get_cluster_groups
        _patch_differences_cluster_compat._applied = True


    def ensure_differences_available():
        """Импортирует differences.ATTgt и применяет слой совместимости 0.3.0 без установки пакетов."""
        if importlib.util.find_spec("differences") is None:
            raise RuntimeError(
                "Package 'differences' is required but not installed. "
                "Install with: pip install differences==0.3.0"
            )
        from differences import ATTgt
        _patch_differences_cluster_compat()
        return ATTgt


    def _filter_kwargs(func, kwargs: dict) -> dict:
        params = inspect.signature(func).parameters
        filtered = {k: v for k, v in kwargs.items() if k in params}
        cluster_var = filtered.get("cluster_var")
        if isinstance(cluster_var, list):
            if len(cluster_var) == 1:
                filtered["cluster_var"] = cluster_var[0]
            elif len(cluster_var) == 0:
                filtered["cluster_var"] = None
        return filtered


    def _numeric_day_index(dates: pd.Series, origin: pd.Timestamp) -> pd.Series:
        return (pd.to_datetime(dates) - origin).dt.days


    CALENDAR_ORIGIN = pd.to_datetime(panel["date"].min()).normalize()


    def prepare_differences_panel(outcome: str, change_type: str, allowed_cohort_dates: list[str]):
        """Строит панель гексагон-день для differences.ATTgt на общем наборе когорт воздействия.

        Повторно использует make_treated_sample / make_control_sample с теми же правилами исходов и ограничивает
        единицы воздействия общими когортами сбалансированного окна. Возвращает (panel, metadata).
        """
        treated = make_treated_sample(outcome, change_type)
        control = make_control_sample(outcome)
        if allowed_cohort_dates:
            allowed = {pd.Timestamp(d) for d in allowed_cohort_dates}
            treated = treated[treated["treatment_date"].isin(allowed)].copy()

        eligible_hex = set(panel["hex"].unique())
        if outcome not in treated.columns or outcome not in control.columns:
            raise KeyError(
                f"Outcome {outcome!r} absent from treated/control order frames. "
                "Load apps from build_analysis_panel (includes t_available)."
            )
        treated = treated[treated["hex"].isin(eligible_hex)].dropna(subset=[outcome]).copy()
        control = control[control["hex"].isin(eligible_hex)].dropna(subset=[outcome]).copy()
        if treated.empty or control.empty:
            raise ValueError(f"Empty treated/control sample for {outcome} x {change_type}")

        # differences резервирует внутренние имена _treated, _control, _cohort и другие.
        # Временный маркер удаляется перед возвратом панели.
        _RESERVED = {"_relative_time", "_cohort", "_treated", "_control", "_after_event", "_w"}
        for frame in (treated, control):
            drop_cols = [c for c in _RESERVED if c in frame.columns]
            if drop_cols:
                frame.drop(columns=drop_cols, inplace=True)

        treated["_differences_treated"] = 1
        control["_differences_treated"] = 0
        stacked = pd.concat([treated, control], ignore_index=True, sort=False)
        if "date" not in stacked.columns:
            stacked["date"] = stacked["request_timestamp"].dt.normalize()

        agg = {
            outcome: outcome_agg_func(outcome),
            "request_timestamp": "count",
            "_differences_treated": "max",
        }
        reg = (
            stacked.groupby(["hex", "date"], as_index=False)
            .agg(agg)
            .rename(columns={"request_timestamp": "n_orders"})
        )
        hex_meta = (
            stacked.sort_values(["hex", "_differences_treated"], ascending=[True, False])
            .groupby("hex", as_index=False)
            .agg(
                cohort_date=("treatment_date", "first"),
                is_treated_attgt=("_differences_treated", "max"),
            )
        )
        reg = reg.merge(hex_meta, on="hex", how="left", validate="many_to_one")
        reg["date"] = _numeric_day_index(
            pd.to_datetime(reg["date"]).dt.normalize(), CALENDAR_ORIGIN
        ).astype(int)
        reg["cohort"] = _numeric_day_index(reg["cohort_date"], CALENDAR_ORIGIN).astype("float")
        reg.loc[reg["is_treated_attgt"] == 0, "cohort"] = np.nan
        reg["cluster_hex"] = reg["hex"]
        # Маркер удаляется, чтобы differences создал собственный _treated.
        reg = reg.drop(columns=["_differences_treated"], errors="ignore")
        reserved_left = [c for c in reg.columns if c in _RESERVED]
        if reserved_left:
            raise ValueError(
                f"ATTgt panel still contains reserved columns {reserved_left} "
                f"for {outcome} x {change_type}"
            )
        if outcome not in reg.columns:
            raise KeyError(
                f"Outcome {outcome!r} missing from differences panel for {change_type}. "
                "Ensure compute_time_variables / analysis-panel orders were used."
            )
        reg = reg.dropna(subset=[outcome]).set_index(["hex", "date"]).sort_index()
        if not reg.index.is_unique:
            raise ValueError(f"differences panel not unique on hex x date for {outcome} x {change_type}")

        cohorts_used = sorted(pd.to_datetime(treated["treatment_date"].dropna().unique()))
        meta = {
            "outcome": outcome,
            "change_type": change_type,
            "n_treated_hex": int(treated["hex"].nunique()),
            "n_control_hex": int(control["hex"].nunique()),
            "cohorts_used": ", ".join(str(pd.Timestamp(c).date()) for c in cohorts_used),
            "origin_date": CALENDAR_ORIGIN,
            "unit": "days" if outcome == SPEED_OUTCOME else "proportion",
            "n_panel_rows": int(len(reg)),
        }
        return reg, meta


    def run_differences_attgt(reg_panel: pd.DataFrame, outcome: str, *, n_jobs: int = 1) -> dict:
        """Оценивает differences.ATTgt и агрегирует по событийному времени, сохраняя модель и кортежи событий."""
        ATTgt = ensure_differences_available()
        init_kwargs = {
            "data": reg_panel, "cohort_name": "cohort", "cohort_column": "cohort",
            "base_period": "universal", "anticipation": 0, "freq": "D",
        }
        model = ATTgt(**_filter_kwargs(ATTgt, init_kwargs))
        fit_kwargs = {
            "formula": outcome, "control_group": "never_treated", "alpha": ALPHA,
            "cluster_var": ["cluster_hex"], "n_jobs": n_jobs, "progress_bar": False,
            "backend": "threading",
        }
        model.fit(**_filter_kwargs(model.fit, fit_kwargs))
        agg_kwargs = _filter_kwargs(
            model.aggregate,
            {"alpha": ALPHA, "cluster_var": ["cluster_hex"], "n_jobs": n_jobs, "backend": "threading"},
        )
        event_df = model.aggregate("event", **agg_kwargs)
        # Именованные кортежи событий сохраняют функции влияния, в отличие от DataFrame.
        event_ntl_dict = model.results(type_of_aggregation="event", to_dataframe=False)
        if isinstance(event_ntl_dict, dict):
            event_ntl = event_ntl_dict.get("full_sample", next(iter(event_ntl_dict.values())))
        else:
            event_ntl = event_ntl_dict
        return {"model": model, "event_df": event_df, "event_ntl": event_ntl}


    # Хранилище оценённых артефактов с ключом (outcome, change_type).
    attgt_artifacts: dict[tuple[str, str], dict] = {}

    if RUN_ATTGT:
        for _key, _info in balanced_windows.items():
            progress.set_postfix(outcome=_key[0], change_type=_key[1], analysis="ATTgt", refresh=False)
            _outcome, _ctype = _key
            if _info["status"] != "ok":
                continue
            try:
                _reg, _meta = prepare_differences_panel(_outcome, _ctype, _info["allowed_cohorts"])
                _res = run_differences_attgt(_reg, _outcome, n_jobs=1)
                attgt_artifacts[_key] = {"meta": _meta, **_res}
                print(f"OK ATTgt: {_outcome} x {_ctype} | treated_hex={_meta['n_treated_hex']} "
                      f"control_hex={_meta['n_control_hex']}")
            except Exception as exc:  # noqa: BLE001 - recorded, not silenced
                failed_models.append({
                    "outcome": _outcome, "change_type": _ctype, "stage": "attgt_fit",
                    "error": repr(exc),
                })
                print(f"FAIL ATTgt: {_outcome} x {_ctype}: {exc}")
            finally:
                progress.update(1)
    else:
        print("[INFO] RUN_ATTGT=False: skipping ATTgt fit; section 7 will attempt to load cached "
              "betahat/sigma from outputs/honest_did if present.")


OK ATTgt: sch_flg x region_only | treated_hex=529 control_hex=3928


OK ATTgt: sch_flg x workmode_only | treated_hex=997 control_hex=3928


OK ATTgt: sch_flg x region_and_workmode | treated_hex=720 control_hex=3928


OK ATTgt: sch_flg x CORE_pooled | treated_hex=1917 control_hex=3928


OK ATTgt: meet_flg x region_only | treated_hex=529 control_hex=3928


...

OK ATTgt: utlz_within_25 x CORE_pooled | treated_hex=1770 control_hex=3888


OK ATTgt: t_available x region_only | treated_hex=529 control_hex=3921


OK ATTgt: t_available x workmode_only | treated_hex=996 control_hex=3921


OK ATTgt: t_available x region_and_workmode | treated_hex=718 control_hex=3921


OK ATTgt: t_available x CORE_pooled | treated_hex=1914 control_hex=3921


## 7. Извлечение / восстановление полных ковариационных матриц

Недельные коэффициенты HonestDiD читаются из **дневной** ATTgt event-агрегации при `relative_period == 7 * week` для каждой недели окна, кроме референса `-1`. Полная ковариация — из **stacked influence functions** (`stack_influence_funcs` + `get_vcv_from_if`), с $\\Sigma = V / n$, чтобы диагональ точно воспроизводила стандартные ошибки пакета. Диагональный shortcut не используется. Cluster/multiplier bootstrap ($\\ge 499$, по умолчанию 999) — только если influence functions отсутствуют или `FORCE_CLUSTER_BOOTSTRAP=True` (по умолчанию выключен).

In [23]:
with cell_progress("Build covariance inputs", total=len(attgt_artifacts), unit="model") as progress:
    from differences.models.attgt.utility_ntl import stack_influence_funcs, get_vcv_from_if


    def _unit_multiplier(unit: str) -> float:
        """Доли переводятся в процентные пункты, длительности остаются в днях."""
        return 1.0 if unit == "days" else 100.0


    def cluster_multiplier_bootstrap_vcv(inf_funcs: np.ndarray, n_boot: int, seed: int) -> np.ndarray:
        """Ковариация множительного bootstrap Радемахера по объединённым функциям влияния.

        Используется только как резервный вариант. inf_funcs имеет размер (n_units, k); возвращается ковариация (k, k)
        в том же масштабе, что get_vcv_from_if(...) / n.
        """
        rng = np.random.default_rng(seed)
        n = inf_funcs.shape[0]
        centered = inf_funcs - inf_funcs.mean(axis=0, keepdims=True)
        draws = np.empty((n_boot, inf_funcs.shape[1]))
        for b in range(n_boot):
            weights = rng.choice([-1.0, 1.0], size=n)
            draws[b] = (weights[:, None] * centered).mean(axis=0)
        return np.cov(draws, rowvar=False)


    def build_honestdid_inputs(key: tuple[str, str]) -> dict:
        """Собирает betahat, Sigma и параметры окна одной модели из артефактов ATTgt."""
        outcome, change_type = key
        info = balanced_windows[key]
        art = attgt_artifacts.get(key)
        if art is None:
            raise RuntimeError(f"No ATTgt artefact available for {outcome} x {change_type}")
        meta = art["meta"]
        unit = meta["unit"]
        mult = _unit_multiplier(unit)

        ntl_by_period = {}
        for nt in art["event_ntl"]:
            rp = getattr(nt, "relative_period", None)
            if rp is not None:
                ntl_by_period[int(rp)] = nt

        weeks = list(info["betahat_weeks"])  # по возрастанию, без референсной недели -1
        selected, sel_weeks, missing = [], [], []
        for w in weeks:
            nt = ntl_by_period.get(7 * w)
            if nt is None or getattr(nt, "ATT", None) is None or getattr(nt, "influence_func", None) is None:
                missing.append(w)
                continue
            selected.append(nt)
            sel_weeks.append(w)
        if missing:
            raise RuntimeError(
                f"missing daily event coefficient/influence function at weeks {missing} "
                f"(relative_period {[7 * w for w in missing]}) for {outcome} x {change_type}"
            )

        betahat = np.array([float(nt.ATT) for nt in selected], dtype=float) * mult
        inf_funcs = stack_influence_funcs(selected)
        inf_funcs = np.asarray(inf_funcs, dtype=float)
        n_units = inf_funcs.shape[0]
        used_bootstrap = False
        if FORCE_CLUSTER_BOOTSTRAP:
            vcv_over_n = cluster_multiplier_bootstrap_vcv(inf_funcs, N_BOOTSTRAP, SEED)
            used_bootstrap = True
        else:
            vcv = get_vcv_from_if(inf_funcs)
            vcv_over_n = np.asarray(vcv, dtype=float) / n_units  # Sigma = V / n в масштабе стандартных ошибок пакета
        sigma = vcv_over_n * (mult ** 2)

        num_pre = int(sum(1 for w in sel_weeks if w < 0))
        num_post = int(sum(1 for w in sel_weeks if w >= 0))
        return {
            "outcome": outcome,
            "change_type": change_type,
            "unit": unit,
            "weeks": sel_weeks,
            "pre_weeks": [w for w in sel_weeks if w < 0],
            "post_weeks": [w for w in sel_weeks if w >= 0],
            "betahat": betahat,
            "sigma": sigma,
            "num_pre": num_pre,
            "num_post": num_post,
            "n_units": int(n_units),
            "used_bootstrap": used_bootstrap,
            "meta": meta,
        }


    def model_output_dir(outcome: str, change_type: str) -> Path:
        """Каталог модели: outputs/honest_did/<outcome>/<change_type>/."""
        d = OUT_DIR / outcome / change_type
        d.mkdir(parents=True, exist_ok=True)
        return d


    def save_honestdid_inputs(inp: dict) -> None:
        """Сохраняет betahat, sigma, порядок коэффициентов и диагностику поддержки одной модели."""
        d = model_output_dir(inp["outcome"], inp["change_type"])
        weeks = list(inp["weeks"])
        order = pd.DataFrame({
            "position": np.arange(len(weeks)),
            "event_time": weeks,
            "period_type": ["pre" if w < 0 else "post" for w in weeks],
            "reference_week_omitted": REFERENCE_WEEK,
        })
        order.to_csv(d / "coefficient_order.csv", index=False)
        se = np.sqrt(np.maximum(np.diag(np.asarray(inp["sigma"], dtype=float)), 0.0))
        coef = pd.DataFrame({
            "event_time": weeks,
            "estimate": inp["betahat"],
            "std_error": se,
            "ci_low": np.asarray(inp["betahat"]) - 1.96 * se,
            "ci_high": np.asarray(inp["betahat"]) + 1.96 * se,
        })
        coef.to_csv(d / "event_study_coefficients.csv", index=False)
        cov = pd.DataFrame(inp["sigma"], index=weeks, columns=weeks)
        cov.to_csv(d / "event_study_covariance.csv")
        key = (inp["outcome"], inp["change_type"])
        info = balanced_windows[key]
        support_rows = []
        for w in info.get("window_weeks", []):
            support_rows.append({
                "outcome": inp["outcome"],
                "change_type": inp["change_type"],
                "event_time": w,
                "n_cohorts": info["n_common_cohorts"],
                "n_hex": info.get("n_hex", np.nan),
                "n_orders": info.get("n_orders", np.nan),
                "included_cohorts": "; ".join(info["allowed_cohorts"]),
            })
        pd.DataFrame(support_rows).to_csv(d / "support_diagnostics.csv", index=False)
        meta = {
            "outcome": inp["outcome"],
            "change_type": inp["change_type"],
            "unit": inp["unit"],
            "weeks": inp["weeks"],
            "num_pre": inp["num_pre"],
            "num_post": inp["num_post"],
            "n_units": inp["n_units"],
            "used_bootstrap": inp["used_bootstrap"],
            "covariance_source": (
                "cluster_bootstrap" if inp["used_bootstrap"] else "differences_influence_functions"
            ),
        }
        (d / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")



    honestdid_inputs: dict[tuple[str, str], dict] = {}
    for _key, _info in balanced_windows.items():
        progress.set_postfix(outcome=_key[0], change_type=_key[1], analysis="covariance", refresh=False)
        if _info["status"] != "ok" or _key not in attgt_artifacts:
            continue
        try:
            _inp = build_honestdid_inputs(_key)
            honestdid_inputs[_key] = _inp
            save_honestdid_inputs(_inp)
            print(f"inputs ready: {_key[0]} x {_key[1]} | k={len(_inp['weeks'])} "
                  f"(pre={_inp['num_pre']}, post={_inp['num_post']}) n={_inp['n_units']}")
        except Exception as exc:  # noqa: BLE001 - recorded, not silenced
            failed_models.append({
                "outcome": _key[0], "change_type": _key[1], "stage": "covariance",
                "error": repr(exc),
            })
            print(f"FAIL covariance: {_key[0]} x {_key[1]}: {exc}")
        finally:
            progress.update(1)

    print("Models with HonestDiD inputs:", len(honestdid_inputs))


inputs ready: sch_flg x region_only | k=11 (pre=7, post=4) n=4446
inputs ready: sch_flg x workmode_only | k=16 (pre=7, post=9) n=4905
inputs ready: sch_flg x region_and_workmode | k=16 (pre=7, post=9) n=4622
inputs ready: sch_flg x CORE_pooled | k=16 (pre=7, post=9) n=5791
inputs ready: meet_flg x region_only | k=11 (pre=7, post=4) n=4446
inputs ready: meet_flg x workmode_only | k=16 (pre=7, post=9) n=4788
inputs ready: meet_flg x region_and_workmode | k=16 (pre=7, post=9) n=4622
inputs ready: meet_flg x CORE_pooled | k=16 (pre=7, post=9) n=5665
inputs ready: success_flg x region_only | k=8 (pre=7, post=1) n=4419
inputs ready: success_flg x workmode_only | k=13 (pre=7, post=6) n=4761
inputs ready: success_flg x region_and_workmode | k=13 (pre=7, post=6) n=4595
inputs ready: success_flg x CORE_pooled | k=13 (pre=7, post=6) n=5638
inputs ready: utlz_within_25 x region_only | k=8 (pre=7, post=1) n=4406
inputs ready: utlz_within_25 x workmode_only | k=13 (pre=7, post=6) n=4747
inputs ready

## 8. Проверка коэффициентов, стандартных ошибок и ковариации

`validate_honestdid_input` проверяет конечность, симметрию, согласованность размерностей (`num_pre + num_post == len(betahat)`) и положительную полуопределённость. Только **крошечные** отрицательные собственные значения (числовой шум) проецируются к ближайшей PSD-матрице; материально не-PSD ковариация вызывает ошибку. Также задаются векторы весов оцениваемого `l_vec` (`on_impact`, `short_run`, `main_post_average` с весами заявок воздействия и `full_supported_post_average`) и сетки чувствительности. Дозовые веса $W_h^+$/$W_h^-$ **не** используются здесь и не должны путаться с параметрами чувствительности $M$/$\bar{M}$.

In [24]:
with cell_progress("Validate HonestDiD inputs", total=len(honestdid_inputs), unit="model") as progress:
    def validate_honestdid_input(betahat, sigma, num_pre, num_post, tol: float = 1e-8):
        """Проверяет входную пару HonestDiD и минимально корректирует PSD. Возвращает (betahat, sigma)."""
        betahat = np.asarray(betahat, dtype=float)
        sigma = np.asarray(sigma, dtype=float)
        k = betahat.size
        if betahat.ndim != 1:
            raise ValueError("betahat must be 1-D")
        if sigma.shape != (k, k):
            raise ValueError(f"sigma shape {sigma.shape} incompatible with betahat length {k}")
        if num_pre + num_post != k:
            raise ValueError(f"num_pre({num_pre}) + num_post({num_post}) != len(betahat)({k})")
        if not np.all(np.isfinite(betahat)):
            raise ValueError("betahat contains non-finite values")
        if not np.all(np.isfinite(sigma)):
            raise ValueError("sigma contains non-finite values")
        sigma = 0.5 * (sigma + sigma.T)
        eig = np.linalg.eigvalsh(sigma)
        scale = max(float(np.max(np.abs(eig))), 1.0)
        min_eig = float(eig.min())
        if min_eig < -tol * scale:
            raise ValueError(
                f"sigma is not PSD (min eigenvalue {min_eig:.3e}, tol {-tol * scale:.3e}); "
                "refusing to project a materially indefinite covariance"
            )
        if min_eig < 0:
            vals, vecs = np.linalg.eigh(sigma)
            vals = np.clip(vals, 0.0, None)
            sigma = 0.5 * ((vecs * vals) @ vecs.T + ((vecs * vals) @ vecs.T).T)
        return betahat, sigma


    def basis_vector(index: int, size: int) -> np.ndarray:
        """Единичный вектор заданной длины с позицией e_index, нумерация с единицы."""
        vec = np.zeros(size, dtype=float)
        vec[index - 1] = 1.0
        return vec


    def post_weights_for(outcome: str, change_type: str, post_weeks: list[int]) -> np.ndarray:
        """Веса заявок воздействия по неделям постпериода, нормированные к единице.

        Для отдельных CORE-типов используется outputs/final/event_study_post_weights.csv при наличии;
        для объединённой выборки и пропусков используются числа заявок воздействия по неделям.
        """
        weights = None
        if post_weights_df is not None and change_type != POOLED_LABEL:
            sub = post_weights_df[
                (post_weights_df["outcome"] == outcome)
                & (post_weights_df["change_type"] == change_type)
            ]
            if not sub.empty:
                wmap = dict(zip(sub["event_week"].astype(int), sub["weight"].astype(float)))
                weights = np.array([wmap.get(int(w), 0.0) for w in post_weeks], dtype=float)
        if weights is None or weights.sum() <= 0:
            treated = _treated_with_relweek(outcome, change_type).dropna(subset=["rel_week"])
            counts = treated.assign(rel_week=treated["rel_week"].astype(int)).groupby("rel_week").size()
            weights = np.array([float(counts.get(int(w), 0)) for w in post_weeks], dtype=float)
        total = weights.sum()
        if total <= 0:
            weights = np.ones(len(post_weeks), dtype=float)
            total = weights.sum()
        return weights / total


    def build_l_vec(estimand: str, inp: dict) -> np.ndarray:
        """Вектор весов постпериода l_vec длины num_post для заданного параметра; сумма равна единице."""
        post_weeks = inp["post_weeks"]
        num_post = inp["num_post"]
        if num_post <= 0:
            raise ValueError("no post periods available for l_vec")
        if estimand == "on_impact":
            vec = basis_vector(1, num_post)  # неделя 0 — первый период после воздействия
        elif estimand == "short_run":
            idx = [i for i, w in enumerate(post_weeks) if 0 <= w <= 3]
            vec = np.zeros(num_post, dtype=float)
            if not idx:
                idx = [0]
            vec[idx] = 1.0 / len(idx)
        elif estimand == "main_post_average":
            vec = post_weights_for(inp["outcome"], inp["change_type"], post_weeks)
        elif estimand == "full_supported_post_average":
            vec = np.ones(num_post, dtype=float) / num_post
        else:
            raise ValueError(f"unknown estimand {estimand!r}")
        s = float(vec.sum())
        if not np.isclose(s, 1.0):
            vec = vec / s
        return vec


    def relative_mbar_grid() -> np.ndarray:
        """Сетка относительной величины Mbar от 0.5 до 3 из 10 точек."""
        return np.round(np.linspace(0.5, 3.0, 10), 4)


    def compute_M_pre(inp: dict) -> float:
        """Калибрует M_pre по максимальной абсолютной второй разности коэффициентов предпериода.

        Референсная неделя -1 имеет коэффициент 0; при невозможности расчёта возвращается 0.0.
        """
        coef = {int(w): float(b) for w, b in zip(inp["weeks"], inp["betahat"])}
        coef[REFERENCE_WEEK] = 0.0
        weeks = sorted(w for w in coef if w <= 0)  # предпериод с референсом и границей момента воздействия
        second_diffs = []
        for w in weeks:
            if (w - 1) in coef and (w + 1) in coef and (w + 1) <= REFERENCE_WEEK + 0:
                second_diffs.append(abs(coef[w + 1] - 2 * coef[w] + coef[w - 1]))
        return float(max(second_diffs)) if second_diffs else 0.0


    def smoothness_M_grid(inp: dict) -> np.ndarray:
        """Сетка гладкости M для каждого исхода привязана к M_pre, с резервом по стандартной ошибке."""
        m_pre = compute_M_pre(inp)
        if m_pre <= 0:
            se = float(np.sqrt(np.max(np.diag(inp["sigma"]))))
            m_pre = max(se, 1e-6)
        grid = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 3.0], dtype=float) * m_pre
        return np.round(np.unique(grid), 6)


    ESTIMANDS = ["on_impact", "short_run", "main_post_average", "full_supported_post_average"]

    # Проверить каждую модель и сохранить проверенные sigma и веса оцениваемых параметров.
    validated_inputs: dict[tuple[str, str], dict] = {}
    for _key, _inp in honestdid_inputs.items():
        progress.set_postfix(outcome=_key[0], change_type=_key[1], analysis="validation", refresh=False)
        try:
            _b, _s = validate_honestdid_input(
                _inp["betahat"], _inp["sigma"], _inp["num_pre"], _inp["num_post"]
            )
            _inp = dict(_inp)
            _inp["betahat"], _inp["sigma"] = _b, _s
            _inp["l_vecs"] = {e: build_l_vec(e, _inp) for e in ESTIMANDS}
            _inp["Mbarvec"] = relative_mbar_grid()
            _inp["Mvec"] = smoothness_M_grid(_inp)
            _inp["M_pre"] = compute_M_pre(_inp)
            validated_inputs[_key] = _inp
            for _e, _lv in _inp["l_vecs"].items():
                assert np.isclose(_lv.sum(), 1.0), f"l_vec {_e} does not sum to 1"
            print(f"validated: {_key[0]} x {_key[1]} | M_pre={_inp['M_pre']:.4g} "
                  f"Mvec={np.round(_inp['Mvec'], 4).tolist()}")
        except Exception as exc:  # noqa: BLE001 - recorded, not silenced
            failed_models.append({
                "outcome": _key[0], "change_type": _key[1], "stage": "validate",
                "error": repr(exc),
            })
            print(f"FAIL validate: {_key[0]} x {_key[1]}: {exc}")
        finally:
            progress.update(1)

    print("Validated models:", len(validated_inputs))

validated: sch_flg x region_only | M_pre=22.95 Mvec=[0.0, 11.4765, 22.9531, 34.4296, 45.9061, 68.8592]
validated: sch_flg x workmode_only | M_pre=13.04 Mvec=[0.0, 6.5211, 13.0422, 19.5632, 26.0843, 39.1265]
validated: sch_flg x region_and_workmode | M_pre=13.84 Mvec=[0.0, 6.918, 13.8359, 20.7539, 27.6718, 41.5078]
validated: sch_flg x CORE_pooled | M_pre=16.6 Mvec=[0.0, 8.3, 16.5999, 24.8999, 33.1998, 49.7997]
validated: meet_flg x region_only | M_pre=34.92 Mvec=[0.0, 17.4586, 34.9172, 52.3758, 69.8345, 104.7517]
validated: meet_flg x workmode_only | M_pre=26.39 Mvec=[0.0, 13.1939, 26.3877, 39.5816, 52.7754, 79.1632]
validated: meet_flg x region_and_workmode | M_pre=15.48 Mvec=[0.0, 7.7392, 15.4785, 23.2177, 30.9569, 46.4354]
validated: meet_flg x CORE_pooled | M_pre=12.3 Mvec=[0.0, 6.1502, 12.3004, 18.4505, 24.6007, 36.9011]
validated: success_flg x region_only | M_pre=42.98 Mvec=[0.0, 21.49, 42.98, 64.47, 85.96, 128.9399]
validated: success_flg x workmode_only | M_pre=22.56 Mvec=[0.0

## 8b. Диагностика предтренда (до HonestDiD)

Отдельные предпериодные коэффициенты, совместный Wald-тест на предпериода ATT, максимальное абсолютное отклонение, первые/вторые разности и `M_pre`. Большое p-value **не** доказывает параллельные тренды — лишь означает, что стандартный тест не отвергает; анализ чувствительности HonestDiD нужен именно потому, что неотвержение не является подтверждением.

In [25]:
with cell_progress("Compute pretrend diagnostics", total=len(validated_inputs), unit="model") as progress:
    def joint_wald_pretrend(betahat: np.ndarray, sigma: np.ndarray, num_pre: int) -> dict:
        """Совместный Wald-тест равенства нулю всех коэффициентов предпериода событийной модели."""
        from scipy import stats

        pre = np.asarray(betahat[:num_pre], dtype=float)
        cov = np.asarray(sigma[:num_pre, :num_pre], dtype=float)
        # стабилизация малых собственных значений для обращения
        eig = np.linalg.eigvalsh(cov)
        if eig.min() < 0:
            cov = cov + (-eig.min() + 1e-12) * np.eye(cov.shape[0])
        try:
            inv = np.linalg.inv(cov)
        except np.linalg.LinAlgError:
            inv = np.linalg.pinv(cov)
        w = float(pre.T @ inv @ pre)
        df = int(num_pre)
        p = float(1.0 - stats.chi2.cdf(w, df))
        return {"wald_stat": w, "df": df, "pvalue": p}


    pretrend_rows = []
    for _key, _inp in validated_inputs.items():
        progress.set_postfix(outcome=_key[0], change_type=_key[1], analysis="pretrend", refresh=False)
        _b = np.asarray(_inp["betahat"], dtype=float)
        _s = np.asarray(_inp["sigma"], dtype=float)
        _np = int(_inp["num_pre"])
        pre = _b[:_np]
        pre_weeks = list(_inp["pre_weeks"])
        first_diff = np.diff(pre) if len(pre) >= 2 else np.array([])
        second_diff = np.diff(pre, n=2) if len(pre) >= 3 else np.array([])
        wald = joint_wald_pretrend(_b, _s, _np)
        se_pre = np.sqrt(np.maximum(np.diag(_s)[:_np], 0.0))
        wide_ci = bool(np.any(1.96 * se_pre > np.maximum(np.abs(pre), 1e-8) * 5))
        row = {
            "outcome": _inp["outcome"],
            "change_type": _inp["change_type"],
            "n_pre_periods": _np,
            "pre_weeks": ",".join(map(str, pre_weeks)),
            "pre_coefficients": ";".join(f"{x:.6g}" for x in pre),
            "max_abs_pre": float(np.max(np.abs(pre))) if len(pre) else np.nan,
            "max_abs_first_diff": float(np.max(np.abs(first_diff))) if len(first_diff) else np.nan,
            "max_abs_second_diff": float(np.max(np.abs(second_diff))) if len(second_diff) else np.nan,
            "M_pre": float(_inp.get("M_pre", np.nan)),
            "wald_stat": wald["wald_stat"],
            "wald_df": wald["df"],
            "pretrend_pvalue": wald["pvalue"],
            "precision_warning_wide_pre_ci": wide_ci,
            "interpretation": (
                "standard test does not reject H0: pre-ATTs=0, but this does not confirm "
                "exact parallel trends; HonestDiD sensitivity analysis is therefore required"
                if wald["pvalue"] > 0.05
                else "standard pretrend test rejects H0; HonestDiD bounds remain informative "
                "and are reported separately from the main confirmed specification when methodology excludes the cohort"
            ),
        }
        pretrend_rows.append(row)
        # значение также сохраняется во входе для сводной таблицы
        _inp["pretrend_pvalue"] = wald["pvalue"]
        _inp["wald_stat"] = wald["wald_stat"]
        progress.update(1)

    pretrend_diagnostics = pd.DataFrame(pretrend_rows)
    pretrend_diagnostics.to_csv(OUT_DIR / "pretrend_diagnostics.csv", index=False)
    display(pretrend_diagnostics)

           outcome          change_type  n_pre_periods             pre_weeks  \
0          sch_flg          region_only              7  -8,-7,-6,-5,-4,-3,-2   
1          sch_flg        workmode_only              7  -8,-7,-6,-5,-4,-3,-2   
2          sch_flg  region_and_workmode              7  -8,-7,-6,-5,-4,-3,-2   
3          sch_flg          CORE_pooled              7  -8,-7,-6,-5,-4,-3,-2   
4         meet_flg          region_only              7  -8,-7,-6,-5,-4,-3,-2   
5         meet_flg        workmode_only              7  -8,-7,-6,-5,-4,-3,-2   
6         meet_flg  region_and_workmode              7  -8,-7,-6,-5,-4,-3,-2   
7         meet_flg          CORE_pooled              7  -8,-7,-6,-5,-4,-3,-2   
8      success_flg          region_only              7  -8,-7,-6,-5,-4,-3,-2   
9      success_flg        workmode_only              7  -8,-7,-6,-5,-4,-3,-2   
10     success_flg  region_and_workmode              7  -8,-7,-6,-5,-4,-3,-2   
11     success_flg          CORE_pooled 

## 9. Базовые графики событийной модели

Базовые коэффициенты ATTgt событийная модель (вход HonestDiD `betahat`) с 95% pointwise интервалами из $\\sqrt{\\mathrm{diag}(\\Sigma)}$ в общем стиле публикации. Доли отображаются в процентных пунктах; `t_available` — в днях. Каждый рисунок сохраняется как PNG 300 dpi и векторный PDF в `figures/honest_did/`.

In [26]:
with cell_progress("Plot baseline estimates", total=len(validated_inputs), unit="model") as progress:
    def plot_baseline_event_study(inp: dict):
        """Строит базовую событийную модель betahat +/- 1.96 SE с референсной неделей в нуле."""
        weeks = list(inp["weeks"])
        betahat = np.asarray(inp["betahat"], dtype=float)
        se = np.sqrt(np.diag(np.asarray(inp["sigma"], dtype=float)))
        unit_label = "days" if inp["unit"] == "days" else "percentage points"

        plot_weeks = weeks + [REFERENCE_WEEK]
        plot_vals = list(betahat) + [0.0]
        plot_se = list(se) + [0.0]
        order = np.argsort(plot_weeks)
        xw = np.array(plot_weeks)[order]
        yv = np.array(plot_vals)[order]
        ye = np.array(plot_se)[order]

        with plot_style():
            fig, ax = plt.subplots(figsize=(6.4, 3.6))
            ax.axhline(0.0, color=PALETTE["zero"], linewidth=0.8)
            ax.axvline(-0.5, color=PALETTE["text_muted"], linewidth=0.7, linestyle=":")
            ax.errorbar(
                xw, yv, yerr=1.96 * ye, fmt="o-", color=PALETTE["treated"],
                ecolor=PALETTE["text_muted"], elinewidth=1.0, capsize=2.5, markersize=4.0,
            )
            ref_x = REFERENCE_WEEK
            ax.scatter([ref_x], [0.0], color=PALETTE["control"], zorder=5, s=28,
                       label=f"reference (week {REFERENCE_WEEK})")
            ax.set_xlabel("Event week (relative to treatment)")
            ax.set_ylabel(f"ATT ({unit_label})")
            ax.set_title(f"{inp['outcome']} x {inp['change_type']} - baseline ATTgt event study")
            ax.legend(loc="best")
            fig.tight_layout()
            stem = f"{inp['outcome']}_{inp['change_type']}_event_study"
            save_figure(fig, FIG_DIR / f"{stem}.pdf", preview_png=True, preview_dpi=300)
        return stem


    baseline_figures = []
    for _key, _inp in validated_inputs.items():
        progress.set_postfix(outcome=_key[0], change_type=_key[1], analysis="baseline_plot", refresh=False)
        try:
            _stem = plot_baseline_event_study(_inp)
            baseline_figures.append(_stem)
        except Exception as exc:  # noqa: BLE001 - recorded, not silenced
            failed_models.append({
                "outcome": _key[0], "change_type": _key[1], "stage": "baseline_plot",
                "error": repr(exc),
            })
            print(f"FAIL baseline plot: {_key[0]} x {_key[1]}: {exc}")
        finally:
            progress.update(1)

    print("Baseline figures written:", len(baseline_figures))


Baseline figures written: 20


## 9b. Контрольные экспорты перед HonestDiD

Сохраняем метаданные сессии, статус реестра моделей и входные артефакты **до** R-моста HonestDiD. Если раздел 10 останавливается из-за отсутствия R-пакетов, эти файлы остаются доступны для инспекции; самодельные границы чувствительности не записываются.

In [27]:
with cell_progress("Write input checkpoints", total=len(validated_inputs) + len(balanced_windows) + 3, unit="item") as progress:
    import importlib.metadata as ilmd


    def _pkg_version_early(name: str) -> str:
        try:
            return ilmd.version(name)
        except Exception:
            return "unknown"


    # Повторное сохранение проверенных коэффициентов модели из раздела 7.
    for _key, _inp in validated_inputs.items():
        save_honestdid_inputs(_inp)
        progress.update(1)

    checkpoint_registry_rows = []
    for _key, _info in balanced_windows.items():
        _o, _c = _key
        if _info["status"] != "ok":
            status = "insufficient_support"
        elif _key not in validated_inputs:
            status = "computation_failed"
        else:
            status = "inputs_ready_honestdid_not_executed"
        spec = next((r for r in OUTCOME_SPECS if r["outcome"] == _o), {})
        checkpoint_registry_rows.append({
            "outcome": _o,
            "change_type": _c,
            "sample_rule": spec.get("sample_rule", ""),
            "cohort_exclusions": (
                f"2022-07-27 {spec.get('cohort_0727', '')}: {spec.get('cohort_0727_reason', '')}"
            ),
            "event_window": (
                f"pre={_info['pre_weeks']}, ref={REFERENCE_WEEK}, post={_info['post_weeks']}"
            ),
            "estimands": ", ".join(ESTIMANDS),
            "status": status,
        })
        progress.update(1)

    checkpoint_registry = pd.DataFrame(checkpoint_registry_rows)
    checkpoint_registry.to_csv(OUT_DIR / "model_registry.csv", index=False)

    failed_models_df = pd.DataFrame(failed_models)
    failed_models_df.to_csv(OUT_DIR / "failed_models.csv", index=False)

    session_info_early = {
        "python_version": sys.version.split()[0],
        "platform": platform.platform(),
        "seed": SEED,
        "numpy": _pkg_version_early("numpy"),
        "pandas": _pkg_version_early("pandas"),
        "differences": _pkg_version_early("differences"),
        "rscript_path": RSCRIPT_PATH,
        "r_version": R_VERSION,
        "honestdid_ready_at_checkpoint": False,
        "run_timestamp_utc": RUN_TIMESTAMP,
        "note": "checkpoint before HonestDiD section 10; may be overwritten after a successful full run",
    }
    (OUT_DIR / "session_info.json").write_text(
        json.dumps(session_info_early, indent=2), encoding="utf-8"
    )
    (OUT_DIR / "session_info.txt").write_text(
        "\n".join(f"{k}: {v}" for k, v in session_info_early.items()) + "\n",
        encoding="utf-8",
    )

    run_manifest_early = {
        "timestamp": RUN_TIMESTAMP,
        "seed": SEED,
        "estimator": "differences.ATTgt (Callaway-SantAnna)",
        "control_group": "never_treated",
        "cluster_variable": "cluster_hex",
        "outcomes": ANALYSIS_OUTCOMES,
        "change_types": CHANGE_TYPES,
        "reference_week": REFERENCE_WEEK,
        "n_validated_inputs": len(validated_inputs),
        "n_failed": len(failed_models),
        "force_cluster_bootstrap": FORCE_CLUSTER_BOOTSTRAP,
        "n_bootstrap": N_BOOTSTRAP,
        "checkpoint": "pre_honestdid_section_10",
    }
    # метаданные Git при наличии
    try:
        import subprocess as _sp

        run_manifest_early["git_commit"] = _sp.check_output(
            ["git", "rev-parse", "HEAD"], cwd=str(PROJECT_ROOT), text=True
        ).strip()
        run_manifest_early["git_branch"] = _sp.check_output(
            ["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=str(PROJECT_ROOT), text=True
        ).strip()
        run_manifest_early["dirty_working_tree"] = bool(
            _sp.check_output(["git", "status", "--porcelain"], cwd=str(PROJECT_ROOT), text=True).strip()
        )
    except Exception as exc:  # noqa: BLE001 — recorded explicitly
        run_manifest_early["git_error"] = repr(exc)

    # хеши входных файлов
    for label, path in (
        ("applications", APPLICATIONS_PATH),
        ("hexagons", HEXAGONS_PATH),
        ("event_study_post_weights", FINAL_DIR / "event_study_post_weights.csv"),
    ):
        try:
            import hashlib

            h = hashlib.sha256()
            with open(path, "rb") as f:
                for chunk in iter(lambda: f.read(1 << 20), b""):
                    h.update(chunk)
            run_manifest_early[f"hash_{label}"] = h.hexdigest()
            run_manifest_early[f"path_{label}"] = str(path)
        except Exception as exc:  # noqa: BLE001
            run_manifest_early[f"hash_{label}_error"] = repr(exc)
        finally:
            progress.update(1)

    (OUT_DIR / "run_manifest.json").write_text(
        json.dumps(run_manifest_early, indent=2), encoding="utf-8"
    )
    print("Checkpoint exports written to", OUT_DIR)
    display(checkpoint_registry)


Checkpoint exports written to <PROJECT_ROOT>/…


           outcome          change_type  \
0          sch_flg          region_only   
1          sch_flg        workmode_only   
2          sch_flg  region_and_workmode   
3          sch_flg          CORE_pooled   
4         meet_flg          region_only   
5         meet_flg        workmode_only   
6         meet_flg  region_and_workmode   
7         meet_flg          CORE_pooled   
8      success_flg          region_only   
9      success_flg        workmode_only   
10     success_flg  region_and_workmode   
11     success_flg          CORE_pooled   
12  utlz_within_25          region_only   
13  utlz_within_25        workmode_only   
14  utlz_within_25  region_and_workmode   
15  utlz_within_25          CORE_pooled   
16     t_available          region_only   
17     t_available        workmode_only   
18     t_available  region_and_workmode   
19     t_available          CORE_pooled   

                                          sample_rule  \
0   CORE treated vs never-treated; all 

## 10. Проверка работоспособности R-моста HonestDiD

Мост записывает `betahat.csv`, `sigma.csv` и `config.json`, запускает временный
R-скрипт с официальными функциями `HonestDiD` и возвращает результаты в CSV.

Для выполнения требуется `Rscript` в `PATH` и R-пакеты `HonestDiD`, `jsonlite`,
`readr`. Если R или хотя бы один пакет недоступен, анализ HonestDiD не
выполняется и ячейка завершается ошибкой. Абсолютные локальные пути и сведения о
машине не сохраняются.


In [28]:
HONESTDID_MODEL_TIMEOUT_SEC = 30 * 60
R_BRIDGE_SCRIPT = OUT_DIR / "honestdid_bridge.R"
R_BRIDGE_SOURCE = r"""
args <- commandArgs(trailingOnly = TRUE)
cfg <- jsonlite::fromJSON(args[1], simplifyVector = TRUE)

flatten_df <- function(df) {
  out <- as.data.frame(df, stringsAsFactors = FALSE)
  for (nm in names(out)) {
    col <- out[[nm]]
    if (is.matrix(col)) {
      out[[nm]] <- as.vector(col)
    } else if (is.list(col)) {
      out[[nm]] <- vapply(col, function(x) {
        if (is.null(x) || length(x) == 0 || (length(x) == 1 && is.na(x))) {
          NA_character_
        } else {
          as.character(x[[1]])
        }
      }, character(1))
    }
  }
  out
}

bounds_df <- function(result) {
  out <- flatten_df(result)
  lower_names <- tolower(names(out))
  lb_idx <- match("lb", lower_names)
  ub_idx <- match("ub", lower_names)
  if (is.na(lb_idx) || is.na(ub_idx)) stop("HonestDiD result lacks lb/ub")
  data.frame(lb = as.numeric(out[[lb_idx]]), ub = as.numeric(out[[ub_idx]]))
}

betahat <- as.numeric(utils::read.csv(cfg$betahat_path)$betahat)
sigma <- as.matrix(utils::read.csv(cfg$sigma_path, check.names = FALSE))
storage.mode(sigma) <- "double"
suppressWarnings(suppressMessages(library(HonestDiD)))
l_vec <- matrix(as.numeric(cfg$l_vec), ncol = 1)
alpha <- as.numeric(cfg$alpha)
num_pre <- as.integer(cfg$num_pre)
num_post <- as.integer(cfg$num_post)
operation <- if (is.null(cfg$operation)) "sensitivity" else as.character(cfg$operation)

cat("STAGE: original\n")
flush.console()
orig <- constructOriginalCS(
  betahat = betahat, sigma = sigma, numPrePeriods = num_pre,
  numPostPeriods = num_post, l_vec = l_vec, alpha = alpha
)

evaluate_values <- function(values) {
  values <- as.numeric(values)
  if (identical(cfg$analysis, "relmag")) {
    result <- createSensitivityResults_relativeMagnitudes(
      betahat = betahat, sigma = sigma, numPrePeriods = num_pre,
      numPostPeriods = num_post, Mbarvec = values, l_vec = l_vec, alpha = alpha
    )
  } else {
    result <- createSensitivityResults(
      betahat = betahat, sigma = sigma, numPrePeriods = num_pre,
      numPostPeriods = num_post, Mvec = values, l_vec = l_vec,
      alpha = alpha, method = cfg$method
    )
  }
  out <- bounds_df(result)
  if (nrow(out) != length(values)) {
    stop(sprintf("Expected %d sensitivity rows, got %d", length(values), nrow(out)))
  }
  out$param_value <- values
  out$includes_zero <- out$lb <= 0 & out$ub >= 0
  out
}

if (identical(operation, "sensitivity")) {
  grid <- as.numeric(cfg$grid)
  utils::write.csv(flatten_df(orig), cfg$orig_out_path, row.names = FALSE)
  if (identical(cfg$analysis, "relmag")) {
    res <- createSensitivityResults_relativeMagnitudes(
      betahat = betahat, sigma = sigma, numPrePeriods = num_pre,
      numPostPeriods = num_post, Mbarvec = grid, l_vec = l_vec, alpha = alpha
    )
  } else {
    res <- createSensitivityResults(
      betahat = betahat, sigma = sigma, numPrePeriods = num_pre,
      numPostPeriods = num_post, Mvec = grid, l_vec = l_vec,
      alpha = alpha, method = cfg$method
    )
  }
  utils::write.csv(flatten_df(res), cfg$out_path, row.names = FALSE)
  cat("HONESTDID_OK\n")
} else if (identical(operation, "breakdown")) {
  started <- proc.time()[["elapsed"]]
  stage <- function(...) {
    cat("STAGE:", paste(..., collapse = " "), "\n")
    flush.console()
  }
  stage("original")
  orig_bounds <- bounds_df(orig)
  orig_lb <- as.numeric(orig_bounds$lb[[1]])
  orig_ub <- as.numeric(orig_bounds$ub[[1]])
  if (!is.finite(orig_lb) || !is.finite(orig_ub)) stop("Original CI is non-finite")

  point_cache <- new.env(parent = emptyenv())
  point_diagnostics <- list()
  honestdid_evaluations <- 0L
  honestdid_function_calls <- 0L
  refinement_points_computed <- 0L
  stable_key <- function(value) sprintf("%.17g", as.numeric(value))
  evaluate_point <- function(value, phase) {
    value <- as.numeric(value)
    key <- stable_key(value)
    if (exists(key, envir = point_cache, inherits = FALSE)) {
      return(get(key, envir = point_cache, inherits = FALSE))
    }
    honestdid_evaluations <<- honestdid_evaluations + 1L
    if (identical(phase, "refinement")) {
      refinement_points_computed <<- refinement_points_computed + 1L
    }
    honestdid_function_calls <<- honestdid_function_calls + 1L
    answer <- tryCatch({
      row <- evaluate_values(value)
      list(
        ok = TRUE, row = row, error = NULL,
        diagnostic = list(
          value = value, phase = phase, status = "ok",
          lb = as.numeric(row$lb[[1]]), ub = as.numeric(row$ub[[1]]),
          includes_zero = isTRUE(row$includes_zero[[1]])
        )
      )
    }, error = function(e) {
      list(
        ok = FALSE, row = NULL, error = conditionMessage(e),
        diagnostic = list(
          value = value, phase = phase, status = "error",
          error = conditionMessage(e)
        )
      )
    })
    point_diagnostics[[length(point_diagnostics) + 1L]] <<- answer$diagnostic
    assign(key, answer, envir = point_cache)
    answer
  }
  fail_with_points <- function(message) {
    details <- jsonlite::toJSON(
      point_diagnostics, auto_unbox = TRUE, null = "null", digits = NA
    )
    stop(sprintf("%s; point_diagnostics=%s", message, details), call. = FALSE)
  }

  baseline_includes_zero <- orig_lb <= 0 && orig_ub >= 0
  coarse <- data.frame(
    param_value = numeric(), lb = numeric(), ub = numeric(), includes_zero = logical()
  )
  refinement_iterations <- 0L
  breakdown_value <- NA_real_
  breakdown_beyond_grid <- FALSE
  conclusion_code <- "computation_failed"
  refine_only <- isTRUE(cfg$refine_only)
  source <- if (refine_only) "r_refinement" else "r_full_search"
  have_bracket <- !is.null(cfg$bracket_lo) && !is.null(cfg$bracket_hi)
  lo <- if (have_bracket) as.numeric(cfg$bracket_lo) else NA_real_
  hi <- if (have_bracket) as.numeric(cfg$bracket_hi) else NA_real_

  if (baseline_includes_zero) {
    breakdown_value <- 0.0
    conclusion_code <- "baseline_inconclusive"
  } else {
    if (!refine_only) {
      coarse_grid <- sort(unique(as.numeric(cfg$coarse_grid)))
      if (length(coarse_grid) < 1 || any(!is.finite(coarse_grid))) {
        stop("coarse_grid must contain finite values")
      }
      stage("coarse", sprintf("points=%d", length(coarse_grid)))
      honestdid_function_calls <- honestdid_function_calls + 1L
      batch <- tryCatch({
        rows <- evaluate_values(coarse_grid)
        if (nrow(rows) != length(coarse_grid)) {
          stop(sprintf(
            "Expected %d vectorized coarse rows, got %d",
            length(coarse_grid), nrow(rows)
          ))
        }
        if (any(!is.finite(rows$lb)) || any(!is.finite(rows$ub))) {
          stop("Vectorized coarse result contains non-finite bounds")
        }
        list(ok = TRUE, rows = rows, error = NULL)
      }, error = function(e) {
        list(ok = FALSE, rows = NULL, error = conditionMessage(e))
      })
      if (isTRUE(batch$ok)) {
        honestdid_evaluations <- honestdid_evaluations + length(coarse_grid)
        answers <- vector("list", length(coarse_grid))
        for (i in seq_along(coarse_grid)) {
          row <- batch$rows[i, , drop = FALSE]
          diagnostic <- list(
            value = coarse_grid[[i]], phase = "coarse", status = "ok",
            evaluation_mode = "vectorized_batch",
            lb = as.numeric(row$lb[[1]]), ub = as.numeric(row$ub[[1]]),
            includes_zero = isTRUE(row$includes_zero[[1]])
          )
          answer <- list(
            ok = TRUE, row = row, error = NULL, diagnostic = diagnostic
          )
          answers[[i]] <- answer
          point_diagnostics[[length(point_diagnostics) + 1L]] <- diagnostic
          assign(stable_key(coarse_grid[[i]]), answer, envir = point_cache)
        }
      } else {
        point_diagnostics[[length(point_diagnostics) + 1L]] <- list(
          value = NULL, phase = "coarse_batch", status = "error",
          evaluation_mode = "vectorized_batch",
          error = batch$error
        )
        stage("coarse fallback", batch$error)
        answers <- lapply(coarse_grid, evaluate_point, phase = "coarse")
      }
      for (i in seq_along(answers)) {
        answer <- answers[[i]]
        if (isTRUE(answer$ok)) {
          coarse <- rbind(coarse, data.frame(
            param_value = coarse_grid[[i]], lb = answer$row$lb[[1]],
            ub = answer$row$ub[[1]],
            includes_zero = isTRUE(answer$row$includes_zero[[1]])
          ))
        }
      }
      ok <- vapply(answers, function(x) isTRUE(x$ok), logical(1))
      included <- which(vapply(answers, function(x) {
        isTRUE(x$ok) && isTRUE(x$row$includes_zero[[1]])
      }, logical(1)))
      if (length(included) == 0) {
        if (any(!ok)) {
          fail_with_points("Cannot conclude robustness because coarse points failed")
        }
        breakdown_value <- max(coarse_grid)
        breakdown_beyond_grid <- TRUE
        conclusion_code <- "robust_at_reported_range"
      } else {
        first <- included[[1]]
        hi <- coarse_grid[[first]]
        if (first == 1L) {
          lo <- 0.0
        } else {
          if (!ok[[first - 1L]] ||
              isTRUE(answers[[first - 1L]]$row$includes_zero[[1]]) ||
              any(!ok[seq_len(first - 1L)])) {
            fail_with_points("Failed point prevents a mathematically valid bracket")
          }
          lo <- coarse_grid[[first - 1L]]
        }
        have_bracket <- TRUE
      }
    }

    if (!identical(conclusion_code, "robust_at_reported_range")) {
      if (!have_bracket || !is.finite(lo) || !is.finite(hi) || lo > hi) {
        stop("A finite ordered breakdown bracket is required")
      }
      max_refine <- as.integer(cfg$n_refine)
      tol <- as.numeric(cfg$tol)
      while ((hi - lo) > tol && refinement_points_computed < max_refine) {
        stage(
          "refinement", sprintf("iteration=%d", refinement_iterations + 1L),
          sprintf("bracket=[%.17g,%.17g]", lo, hi)
        )
        mid <- lo + (hi - lo) / 2
        if (exists(stable_key(mid), envir = point_cache, inherits = FALSE)) {
          fail_with_points("Duplicate midpoint cannot shrink refinement bracket")
        }
        answer <- evaluate_point(mid, "refinement")
        refinement_iterations <- refinement_iterations + 1L
        if (!isTRUE(answer$ok)) {
          fail_with_points(sprintf("HonestDiD failed at midpoint %.17g", mid))
        }
        if (isTRUE(answer$row$includes_zero[[1]])) hi <- mid else lo <- mid
      }
      breakdown_value <- hi
      conclusion_code <- "sign_sensitive_to_moderate_violations"
    }
  }

  if (!is.finite(breakdown_value)) stop("Breakdown value is non-finite")
  if (!(conclusion_code %in% c(
    "baseline_inconclusive", "sign_sensitive_to_moderate_violations",
    "robust_at_reported_range"
  ))) stop("Invalid conclusion code")
  payload <- list(
    orig_lb = orig_lb, orig_ub = orig_ub,
    breakdown_value = breakdown_value,
    conclusion_code = conclusion_code,
    breakdown_beyond_grid = breakdown_beyond_grid,
    coarse_diagnostics = list(
      grid = as.numeric(coarse$param_value), lb = as.numeric(coarse$lb),
      ub = as.numeric(coarse$ub), includes_zero = as.logical(coarse$includes_zero)
    ),
    point_diagnostics = point_diagnostics,
    refinement_iterations = refinement_iterations,
    source = source, rscript_calls = 1L,
    honestdid_evaluations = honestdid_evaluations,
    honestdid_function_calls = honestdid_function_calls,
    coarse_points_reused = if (is.null(cfg$coarse_points_reused)) {
      0L
    } else {
      as.integer(cfg$coarse_points_reused)
    },
    refinement_points_computed = refinement_points_computed,
    elapsed_seconds = as.numeric(proc.time()[["elapsed"]] - started),
    status = "completed"
  )
  stage("write result")
  final_path <- cfg$breakdown_out_path
  staging_path <- paste0(final_path, ".tmp.", Sys.getpid())
  if (file.exists(staging_path)) unlink(staging_path)
  jsonlite::write_json(
    payload, staging_path, auto_unbox = TRUE, null = "null", na = "null", digits = NA
  )
  if (!file.exists(staging_path) || file.info(staging_path)$size <= 0) {
    stop("Atomic breakdown JSON staging file is missing or empty")
  }
  jsonlite::fromJSON(staging_path, simplifyVector = FALSE)
  if (file.exists(final_path)) unlink(final_path)
  if (!file.rename(staging_path, final_path)) stop("Atomic breakdown JSON rename failed")
  cat("HONESTDID_BREAKDOWN_OK\n")
  flush.console()
} else {
  stop(sprintf("Unknown bridge operation: %s", operation))
}
"""


def probe_r_packages(rscript: str) -> dict:
    """Возвращает доступность пакетов с явным декодированием вывода подпроцесса."""
    pkgs = ["HonestDiD", "jsonlite", "readr"]
    expr = (
        'cat(paste(sapply(c("HonestDiD","jsonlite","readr"),'
        'function(p) paste0(p,"=",requireNamespace(p, quietly=TRUE))), collapse=";"))'
    )
    proc = subprocess.run([rscript, "-e", expr], capture_output=True, text=False)
    stdout = decode_subprocess_output(proc.stdout)
    _ = decode_subprocess_output(proc.stderr)
    status = {p: False for p in pkgs}
    for entry in stdout.strip().split(";"):
        if "=" in entry:
            name, val = entry.split("=", 1)
            status[name.strip()] = val.strip().upper() == "TRUE"
    return status


def _install_instructions(status: dict) -> str:
    missing = [p for p, ok in status.items() if not ok] or ["HonestDiD", "jsonlite", "readr"]
    quoted = ", ".join(f'"{p}"' for p in missing)
    return (
        "In an R session run:\n"
        f'    install.packages(c({quoted}))\n'
        '    # HonestDiD may require: remotes::install_github("asheshrambachan/HonestDiD")\n'
    )



class HonestDiDBridgeError(RuntimeError):
    """Структурированные метаданные неуспешного вызова R-моста."""

    def __init__(
        self, message, *, stage="bridge", returncode=None, stdout_tail="",
        stderr_tail="", config_path="", workdir="", elapsed_seconds=np.nan,
        rscript_calls=1, honestdid_evaluations=0,
        honestdid_function_calls=0,
    ):
        super().__init__(message)
        self.stage = stage
        self.returncode = returncode
        self.stdout_tail = stdout_tail
        self.stderr_tail = stderr_tail
        self.config_path = str(config_path)
        self.workdir = str(workdir)
        self.elapsed_seconds = elapsed_seconds
        self.rscript_calls = rscript_calls
        self.honestdid_evaluations = honestdid_evaluations
        self.honestdid_function_calls = honestdid_function_calls


def _output_tail(value, limit: int = 6000) -> str:
    text = decode_subprocess_output(value) if isinstance(value, bytes) else str(value or "")
    meaningful = "\n".join(line for line in text.splitlines() if line.strip())
    return meaningful[-limit:]


def _bridge_stage(stdout: str, default: str = "bridge") -> str:
    markers = [
        line.split("STAGE:", 1)[1].strip()
        for line in stdout.splitlines() if "STAGE:" in line
    ]
    return markers[-1] if markers else default


def _structured_bridge_error(
    message, *, returncode, stdout, stderr, config_path, work,
    elapsed_seconds, stage=None, honestdid_evaluations=0,
    honestdid_function_calls=0,
):
    stdout_tail, stderr_tail = _output_tail(stdout), _output_tail(stderr)
    return HonestDiDBridgeError(
        message, stage=stage or _bridge_stage(stdout_tail),
        returncode=returncode, stdout_tail=stdout_tail, stderr_tail=stderr_tail,
        config_path=config_path, workdir=work, elapsed_seconds=elapsed_seconds,
        rscript_calls=1, honestdid_evaluations=honestdid_evaluations,
        honestdid_function_calls=honestdid_function_calls,
    )


def _bridge_error_message(
    *, returncode, analysis: str, tag: str, stdout: str, stderr: str,
    config_path: Path, work: Path,
) -> str:
    return (
        f"HonestDiD R bridge failed: returncode={returncode}; analysis={analysis}; tag={tag}; "
        f"config.json={config_path}; bridge_workdir={work}\n"
        f"stdout:\n{stdout}\nstderr:\n{stderr}"
    )


def _validated_bridge_csv(path: Path, label: str) -> pd.DataFrame:
    if not path.exists() or path.stat().st_size == 0:
        raise ValueError(f"{label} CSV missing or empty: {path}")
    frame = pd.read_csv(path)
    lower = {str(c).lower(): c for c in frame.columns}
    if not {"lb", "ub"}.issubset(lower):
        raise ValueError(f"{label} CSV lacks lb/ub columns: {list(frame.columns)}")
    lb = pd.to_numeric(frame[lower["lb"]], errors="coerce")
    ub = pd.to_numeric(frame[lower["ub"]], errors="coerce")
    if lb.isna().all() or ub.isna().all():
        raise ValueError(f"{label} CSV has all-NaN lb or ub")
    return frame


def _stop_bridge_process(proc: subprocess.Popen) -> tuple[bytes, bytes]:
    """Завершает R-мост, при необходимости принудительно, и ожидает процесс."""
    if proc.poll() is None:
        proc.terminate()
    try:
        return proc.communicate(timeout=3.0)
    except subprocess.TimeoutExpired:
        if proc.poll() is None:
            proc.kill()
        return proc.communicate()


def _launch_bridge(
    *, analysis: str, tag: str, config_path: Path, work: Path,
    timeout_sec: float | None,
) -> tuple[subprocess.Popen, str, str, float]:
    started = time.perf_counter()
    proc = subprocess.Popen(
        [RSCRIPT_PATH, str(R_BRIDGE_SCRIPT), str(config_path)],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=False,
    )
    try:
        stdout_b, stderr_b = proc.communicate(
            timeout=float(timeout_sec) if timeout_sec and timeout_sec > 0 else None
        )
    except KeyboardInterrupt:
        _stop_bridge_process(proc)
        raise
    except subprocess.TimeoutExpired as exc:
        stdout_b, stderr_b = _stop_bridge_process(proc)
        stdout, stderr = map(decode_subprocess_output, (stdout_b, stderr_b))
        raise _structured_bridge_error(
            f"HonestDiD R bridge timed out: analysis={analysis}; tag={tag}",
            returncode="timeout", stdout=stdout, stderr=stderr,
            config_path=config_path, work=work,
            elapsed_seconds=time.perf_counter() - started,
            stage=_bridge_stage(stdout, "timeout"),
        ) from exc
    return (
        proc, decode_subprocess_output(stdout_b),
        decode_subprocess_output(stderr_b), time.perf_counter() - started,
    )


def _write_bridge_inputs(work: Path, betahat, sigma) -> tuple[Path, Path]:
    work.mkdir(parents=True, exist_ok=True)
    betahat, sigma = np.asarray(betahat, dtype=float), np.asarray(sigma, dtype=float)
    betahat_path, sigma_path = work / "betahat.csv", work / "sigma.csv"
    pd.DataFrame({"betahat": betahat}).to_csv(betahat_path, index=False)
    pd.DataFrame(sigma, columns=[f"V{j}" for j in range(sigma.shape[1])]).to_csv(
        sigma_path, index=False
    )
    return betahat_path, sigma_path


def _ensure_bridge_script() -> None:
    if (
        not R_BRIDGE_SCRIPT.exists()
        or R_BRIDGE_SCRIPT.read_text(encoding="utf-8") != R_BRIDGE_SOURCE
    ):
        R_BRIDGE_SCRIPT.write_text(R_BRIDGE_SOURCE, encoding="utf-8")


def run_honestdid(
    analysis: str, betahat, sigma, num_pre: int, num_post: int,
    l_vec, grid, *, method: str = "FLCI", alpha: float = ALPHA,
    tag: str = "smoke", timeout_sec: float | None = None,
) -> dict:
    """Запускает одно R-задание модели с полной заданной сеткой."""
    if not HONESTDID_READY:
        raise RuntimeError("HonestDiD bridge is not ready; see install instructions.")
    timeout_sec = (
        float(globals().get("HONESTDID_MODEL_TIMEOUT_SEC", 30 * 60))
        if timeout_sec is None else timeout_sec
    )
    work = OUT_DIR / "_bridge" / tag
    betahat_path, sigma_path = _write_bridge_inputs(work, betahat, sigma)
    out_path, orig_path = work / "sensitivity.csv", work / "original.csv"
    for stale in (out_path, orig_path):
        stale.unlink(missing_ok=True)
    config = {
        "operation": "sensitivity", "analysis": analysis,
        "betahat_path": betahat_path.as_posix(), "sigma_path": sigma_path.as_posix(),
        "out_path": out_path.as_posix(), "orig_out_path": orig_path.as_posix(),
        "num_pre": int(num_pre), "num_post": int(num_post),
        "l_vec": [float(x) for x in np.asarray(l_vec, dtype=float)],
        "grid": [float(x) for x in np.asarray(grid, dtype=float)],
        "method": method, "alpha": float(alpha),
    }
    config_path = work / "config.json"
    config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
    _ensure_bridge_script()
    proc, stdout, stderr, bridge_elapsed = _launch_bridge(
        analysis=analysis, tag=tag, config_path=config_path,
        work=work, timeout_sec=timeout_sec,
    )
    try:
        if proc.returncode != 0 or "HONESTDID_OK" not in stdout:
            raise RuntimeError("nonzero return code or missing HONESTDID_OK marker")
        sensitivity = _validated_bridge_csv(out_path, "sensitivity")
        original = _validated_bridge_csv(orig_path, "original")
    except Exception as exc:
        raise RuntimeError(_bridge_error_message(
            returncode=proc.returncode, analysis=analysis, tag=tag, stdout=stdout,
            stderr=f"{stderr}\nvalidation_error={exc!r}",
            config_path=config_path, work=work,
        )) from exc
    return {"sensitivity": sensitivity, "original": original}


def run_honestdid_breakdown(
    analysis: str, betahat, sigma, num_pre: int, num_post: int,
    l_vec, coarse_grid, *, method: str, n_refine: int = 25,
    tol: float = 1e-3, alpha: float = ALPHA, refine_only: bool = False,
    bracket_lo: float | None = None, bracket_hi: float | None = None,
    coarse_points_reused: int = 0, tag: str = "breakdown",
    timeout_sec: float | None = None,
) -> dict:
    """Запускает один процесс R.

    honestdid_evaluations считает уникальные проверенные значения параметра чувствительности
    или рассчитанные значения без constructOriginalCS. honestdid_function_calls считает
    вызовы API чувствительности, включая неуспешный пакетный вызов перед поэлементным резервом.
    """
    if not HONESTDID_READY:
        raise RuntimeError("HonestDiD bridge is not ready; see install instructions.")
    timeout_sec = (
        float(globals().get("HONESTDID_MODEL_TIMEOUT_SEC", 30 * 60))
        if timeout_sec is None else timeout_sec
    )
    if refine_only and (bracket_lo is None or bracket_hi is None):
        raise ValueError("refine_only requires bracket_lo and bracket_hi")
    work = OUT_DIR / "_bridge" / tag
    betahat_path, sigma_path = _write_bridge_inputs(work, betahat, sigma)
    result_path = work / "breakdown.json"
    result_path.unlink(missing_ok=True)
    config = {
        "operation": "breakdown", "analysis": analysis,
        "betahat_path": betahat_path.as_posix(), "sigma_path": sigma_path.as_posix(),
        "num_pre": int(num_pre), "num_post": int(num_post),
        "l_vec": [float(x) for x in np.asarray(l_vec, dtype=float)],
        "alpha": float(alpha), "method": method,
        "coarse_grid": [float(x) for x in np.asarray(coarse_grid, dtype=float)],
        "refine_only": bool(refine_only), "n_refine": int(n_refine),
        "tol": float(tol), "coarse_points_reused": int(coarse_points_reused),
        "breakdown_out_path": result_path.as_posix(),
    }
    if bracket_lo is not None:
        config["bracket_lo"] = float(bracket_lo)
    if bracket_hi is not None:
        config["bracket_hi"] = float(bracket_hi)
    config_path = work / "config.json"
    config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
    _ensure_bridge_script()
    proc, stdout, stderr, bridge_elapsed = _launch_bridge(
        analysis=analysis, tag=tag, config_path=config_path,
        work=work, timeout_sec=timeout_sec,
    )
    payload = None
    try:
        if proc.returncode != 0 or "HONESTDID_BREAKDOWN_OK" not in stdout:
            raise ValueError("nonzero return code or missing breakdown marker")
        if not result_path.exists() or result_path.stat().st_size == 0:
            raise ValueError(f"breakdown JSON missing or empty: {result_path}")
        payload = json.loads(result_path.read_text(encoding="utf-8"))
        required = {
            "orig_lb", "orig_ub", "breakdown_value", "conclusion_code",
            "breakdown_beyond_grid", "coarse_diagnostics", "point_diagnostics",
            "refinement_iterations", "source", "rscript_calls",
            "honestdid_evaluations", "honestdid_function_calls",
            "coarse_points_reused", "refinement_points_computed",
            "elapsed_seconds", "status",
        }
        if not required.issubset(payload):
            raise ValueError(f"breakdown JSON lacks {sorted(required - set(payload))}")
        allowed = {
            "baseline_inconclusive", "sign_sensitive_to_moderate_violations",
            "robust_at_reported_range",
        }
        if payload["status"] != "completed" or payload["conclusion_code"] not in allowed:
            raise ValueError("invalid status or conclusion")
        numeric = [
            payload["orig_lb"], payload["orig_ub"], payload["breakdown_value"],
            payload["elapsed_seconds"],
        ]
        if not np.isfinite(np.asarray(numeric, dtype=float)).all():
            raise ValueError("non-finite original CI, conclusion value, or elapsed time")
        if int(payload["rscript_calls"]) != 1:
            raise ValueError("bridge result must report one Rscript call")
        if (
            int(payload["honestdid_evaluations"]) < 0
            or int(payload["honestdid_function_calls"]) < 0
        ):
            raise ValueError("negative HonestDiD diagnostic counter")
    except Exception as exc:
        evaluations = (
            int(payload.get("honestdid_evaluations", 0))
            if isinstance(payload, dict) else 0
        )
        function_calls = (
            int(payload.get("honestdid_function_calls", 0))
            if isinstance(payload, dict) else 0
        )
        raise _structured_bridge_error(
            f"HonestDiD breakdown bridge failed: {exc}",
            returncode=proc.returncode, stdout=stdout,
            stderr=f"{stderr}\nvalidation_error={exc!r}",
            config_path=config_path, work=work, elapsed_seconds=bridge_elapsed,
            honestdid_evaluations=evaluations,
            honestdid_function_calls=function_calls,
        ) from exc
    payload["_bridge_metadata"] = {
        "returncode": int(proc.returncode),
        "stdout_tail": _output_tail(stdout),
        "stderr_tail": _output_tail(stderr),
        "config_path": str(config_path),
        "workdir": str(work),
        "bridge_elapsed": float(bridge_elapsed),
    }
    return payload


with cell_progress("Probe HonestDiD bridge", total=1, unit="probe") as progress:
    if RSCRIPT_PATH is not None and R_VERSION is not None:
        R_PACKAGE_STATUS = probe_r_packages(RSCRIPT_PATH)
    else:
        R_PACKAGE_STATUS = {"HonestDiD": False, "jsonlite": False, "readr": False}
    HONESTDID_READY = RSCRIPT_PATH is not None and all(
        R_PACKAGE_STATUS.get(p, False) for p in ("HonestDiD", "jsonlite", "readr")
    )
    print("R package status:", R_PACKAGE_STATUS)
    print("HONESTDID_READY:", HONESTDID_READY)
    if HONESTDID_READY:
        smoke_beta = np.array([0.0, 0.0, 0.10], dtype=float)
        smoke_sigma = np.array(
            [[0.02, 0.005, 0.004], [0.005, 0.02, 0.006], [0.004, 0.006, 0.03]]
        )
        smoke = run_honestdid(
            "relmag", smoke_beta, smoke_sigma, num_pre=2, num_post=1,
            l_vec=np.array([1.0]), grid=np.array([0.5, 1.0, 1.5]), tag="smoke",
        )
        print("HonestDiD runtime check passed.")
    else:
        print(_install_instructions(R_PACKAGE_STATUS))
        raise RuntimeError(
            "HonestDiD / jsonlite / readr are not installed (and/or Rscript not found)."
        )


R package status: {'HonestDiD': True, 'jsonlite': True, 'readr': True}
HONESTDID_READY: True
HonestDiD runtime check passed.


## 11. Чувствительность по относительной величине ($\\Delta^{RM}$)

Для каждой проверенной модели и estimand запускается `createSensitivityResults_relativeMagnitudes` по сетке $\bar{M}$ `0.5…3` (10 точек) в паре с базовым доверительным множеством из `constructOriginalCS`. $\bar{M}$ — отношение допустимого post-воздействие нарушения параллельных трендов к максимальному наблюдаемому предтренду; это **не** дозовой вес. Назначается формальный `conclusion_code`; эффекты описываются как диапазоны устойчивости (проверка устойчивости), а не как подтверждённые гипотезы.

In [29]:
import gc

HONESTDID_MODEL_TIMEOUT_SEC = int(globals().get("HONESTDID_MODEL_TIMEOUT_SEC", 30 * 60))
SENSITIVITY_KEY_COLS = ["outcome", "change_type", "estimand", "analysis"]
RESULT_KEY_COLS = SENSITIVITY_KEY_COLS + ["param_value"]
REQUIRED_SENSITIVITY_COLS = {
    "lb", "ub", "param_value", "method", "param_name", "outcome",
    "change_type", "estimand", "analysis", "orig_lb", "orig_ub",
    "includes_zero", "unit",
}
analysis_job_status: dict[str, pd.DataFrame] = {}


def _lower_cols(df: pd.DataFrame) -> dict:
    return {str(c).lower(): c for c in df.columns}


def normalize_sensitivity_df(df: pd.DataFrame, param_name: str) -> pd.DataFrame:
    """Нормализует вывод HonestDiD; параметры чувствительности записываются как param_value."""
    lc = _lower_cols(df)
    if not {"lb", "ub"}.issubset(lc):
        raise ValueError(f"HonestDiD output lacks lb/ub: {list(df.columns)}")
    out = pd.DataFrame({
        "lb": pd.to_numeric(df[lc["lb"]], errors="coerce"),
        "ub": pd.to_numeric(df[lc["ub"]], errors="coerce"),
    })
    target = param_name.lower()
    source = lc.get(target)
    if source is None:
        source = lc.get("mbar" if target == "mbar" else "m")
    if source is None:
        raise ValueError(f"HonestDiD output lacks {param_name}: {list(df.columns)}")
    out["param_value"] = pd.to_numeric(df[source], errors="coerce")
    out["method"] = df[lc["method"]] if "method" in lc else "unknown"
    out["param_name"] = param_name
    return out


def original_ci(df: pd.DataFrame) -> tuple[float, float]:
    lc = _lower_cols(df)
    return (
        float(pd.to_numeric(df[lc["lb"]], errors="coerce").iloc[0]),
        float(pd.to_numeric(df[lc["ub"]], errors="coerce").iloc[0]),
    )


def ci_includes_zero(lb: float, ub: float) -> bool:
    return bool(lb <= 0.0 <= ub)


def classify_conclusion(orig_lb: float, orig_ub: float, sens: pd.DataFrame) -> str:
    if ci_includes_zero(orig_lb, orig_ub):
        return "baseline_inconclusive"
    includes = sens["includes_zero"] if "includes_zero" in sens else (
        sens["lb"].le(0.0) & sens["ub"].ge(0.0)
    )
    return (
        "sign_sensitive_to_moderate_violations"
        if bool(includes.any())
        else "robust_at_reported_range"
    )


def _dedup_failed_models() -> pd.DataFrame:
    global failed_models
    cols = ["outcome", "change_type", "estimand", "stage", "error"]
    frame = pd.DataFrame(failed_models)
    for col in cols:
        if col not in frame:
            frame[col] = ""
    frame = frame.drop_duplicates(cols, keep="last").reset_index(drop=True)
    failed_models = frame.to_dict("records")
    frame.to_csv(OUT_DIR / "failed_models.csv", index=False)
    return frame


def _record_failure(outcome: str, change_type: str, estimand: str, stage: str, exc) -> None:
    failed_models.append({
        "outcome": outcome,
        "change_type": change_type,
        "estimand": estimand,
        "stage": stage,
        "error": repr(exc),
    })
    _dedup_failed_models()


def _analysis_jobs(analysis: str) -> list[tuple[tuple[str, str], dict, str]]:
    return [
        (key, inp, estimand)
        for key, inp in validated_inputs.items()
        for estimand in ESTIMANDS
    ]


def _analysis_checkpoint_path(
    analysis: str, outcome: str, change_type: str, estimand: str,
) -> Path:
    directory = OUT_DIR / "checkpoints" / analysis
    directory.mkdir(parents=True, exist_ok=True)
    return directory / f"{analysis}__{outcome}__{change_type}__{estimand}.csv"


def _checkpoint_frame(
    analysis: str, outcome: str, change_type: str, estimand: str,
) -> pd.DataFrame | None:
    path = _analysis_checkpoint_path(analysis, outcome, change_type, estimand)
    if not path.exists() or path.stat().st_size == 0:
        return None
    try:
        frame = pd.read_csv(path)
    except Exception:
        return None
    if frame.empty or not REQUIRED_SENSITIVITY_COLS.issubset(frame.columns):
        return None
    expected = (outcome, change_type, estimand, analysis)
    keys = frame[SENSITIVITY_KEY_COLS].drop_duplicates()
    if len(keys) != 1 or tuple(keys.iloc[0]) != expected:
        return None
    lb = pd.to_numeric(frame["lb"], errors="coerce")
    ub = pd.to_numeric(frame["ub"], errors="coerce")
    if lb.isna().all() or ub.isna().all():
        return None
    inp = validated_inputs.get((outcome, change_type))
    if inp is None or estimand not in ESTIMANDS:
        return None
    expected_grid = np.asarray(
        inp["Mbarvec"] if analysis == "relmag" else inp["Mvec"], dtype=float
    )
    actual_grid = np.sort(pd.to_numeric(frame["param_value"], errors="coerce").dropna().unique())
    if len(actual_grid) != len(expected_grid) or not np.allclose(
        actual_grid, np.sort(expected_grid), rtol=1e-10, atol=1e-12
    ):
        return None
    if not np.isfinite(frame[["orig_lb", "orig_ub"]].to_numpy(dtype=float)).all():
        return None
    return frame


def _decorate_sensitivity(
    raw_sensitivity: pd.DataFrame, raw_original: pd.DataFrame,
    inp: dict, estimand: str, analysis: str,
) -> dict:
    param_name = "Mbar" if analysis == "relmag" else "M"
    sens = normalize_sensitivity_df(raw_sensitivity, param_name)
    orig_lb, orig_ub = original_ci(raw_original)
    sens["outcome"] = inp["outcome"]
    sens["change_type"] = inp["change_type"]
    sens["estimand"] = estimand
    sens["analysis"] = analysis
    sens["orig_lb"] = orig_lb
    sens["orig_ub"] = orig_ub
    sens["includes_zero"] = sens["lb"].le(0.0) & sens["ub"].ge(0.0)
    sens["unit"] = inp["unit"]
    return {
        "table": sens,
        "conclusion_code": classify_conclusion(orig_lb, orig_ub, sens),
        "orig_lb": orig_lb,
        "orig_ub": orig_ub,
    }


def run_sensitivity_for_model(
    inp: dict, estimand: str, analysis: str, method: str = "FLCI",
    *, timeout_sec: float | None = None,
) -> dict:
    grid = inp["Mbarvec"] if analysis == "relmag" else inp["Mvec"]
    result = run_honestdid(
        analysis, inp["betahat"], inp["sigma"], inp["num_pre"], inp["num_post"],
        l_vec=inp["l_vecs"][estimand], grid=grid, method=method,
        tag=f"{analysis}__{inp['outcome']}__{inp['change_type']}__{estimand}",
        timeout_sec=timeout_sec,
    )
    return _decorate_sensitivity(
        result["sensitivity"], result["original"], inp, estimand, analysis
    )


def _rebuild_analysis_state(analysis: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Восстанавливает данные только из валидных контрольных точек текущего реестра."""
    tables, conclusions = [], []
    for _, inp, estimand in _analysis_jobs(analysis):
        frame = _checkpoint_frame(
            analysis, inp["outcome"], inp["change_type"], estimand
        )
        if frame is None:
            continue
        tables.append(frame)
        orig_lb = float(frame["orig_lb"].iloc[0])
        orig_ub = float(frame["orig_ub"].iloc[0])
        conclusions.append({
            "outcome": inp["outcome"],
            "change_type": inp["change_type"],
            "estimand": estimand,
            "analysis": analysis,
            "conclusion_code": classify_conclusion(orig_lb, orig_ub, frame),
            "orig_lb": orig_lb,
            "orig_ub": orig_ub,
            "M_pre": inp.get("M_pre", np.nan),
        })
    results = pd.concat(tables, ignore_index=True) if tables else pd.DataFrame(
        columns=sorted(REQUIRED_SENSITIVITY_COLS)
    )
    if not results.empty:
        results = results.drop_duplicates(RESULT_KEY_COLS, keep="last").reset_index(drop=True)
    conclusion_df = pd.DataFrame(conclusions)
    if not conclusion_df.empty:
        conclusion_df = conclusion_df.drop_duplicates(SENSITIVITY_KEY_COLS, keep="last")
    return results, conclusion_df


def _analysis_failure_path(analysis: str) -> Path:
    return OUT_DIR / (
        "relative_magnitude_failures.csv" if analysis == "relmag"
        else "smoothness_failures.csv"
    )


def _guard_analysis_state(
    analysis: str, results: pd.DataFrame, conclusions: pd.DataFrame,
    status: pd.DataFrame,
) -> None:
    expected = {
        (inp["outcome"], inp["change_type"], estimand, analysis)
        for _, inp, estimand in _analysis_jobs(analysis)
    }
    successes = status.loc[
        status["status"].isin(["completed", "checkpoint"]), SENSITIVITY_KEY_COLS
    ] if not status.empty else pd.DataFrame(columns=SENSITIVITY_KEY_COLS)
    success_keys = set(map(tuple, successes.drop_duplicates().to_numpy()))
    result_keys = (
        set(map(tuple, results[SENSITIVITY_KEY_COLS].drop_duplicates().to_numpy()))
        if not results.empty else set()
    )
    conclusion_keys = (
        set(map(tuple, conclusions[SENSITIVITY_KEY_COLS].drop_duplicates().to_numpy()))
        if not conclusions.empty else set()
    )
    if success_keys and results.empty:
        raise RuntimeError(f"{analysis}: successful jobs exist but results are empty")
    if result_keys != success_keys or conclusion_keys != success_keys:
        raise RuntimeError(
            f"{analysis}: state mismatch success={len(success_keys)}, "
            f"results={len(result_keys)}, conclusions={len(conclusion_keys)}"
        )
    if not results.empty and results.duplicated(RESULT_KEY_COLS).any():
        raise RuntimeError(f"{analysis}: duplicate result keys")
    failed_keys = set(
        map(tuple, status.loc[status["status"] == "failed", SENSITIVITY_KEY_COLS].to_numpy())
    ) if not status.empty else set()
    missing = expected - success_keys - failed_keys
    if len(success_keys) + len(failed_keys) != len(expected) or missing:
        raise RuntimeError(
            f"{analysis} partial-state accounting error: expected={len(expected)}, "
            f"completed={len(success_keys)}, failed={len(failed_keys)}, "
            f"missing_keys={sorted(missing)}; failures_csv={_analysis_failure_path(analysis)}"
        )
    if not success_keys:
        raise RuntimeError(
            f"{analysis}: all {len(expected)} jobs failed; "
            f"failures_csv={_analysis_failure_path(analysis)}"
        )
    if failed_keys:
        print(
            f"{analysis}: partial completion expected={len(expected)} "
            f"completed={len(success_keys)} failed={len(failed_keys)} "
            f"missing_keys=[] failures_csv={_analysis_failure_path(analysis)}"
        )


def run_analysis_jobs(analysis: str, method: str):
    jobs = _analysis_jobs(analysis)
    if not jobs:
        raise RuntimeError(f"validated_inputs is empty; cannot run {analysis}")
    status_rows, failure_rows = [], []
    completed = failed = checkpoints = 0
    started = time.perf_counter()
    try:
        with cell_progress(
            f"{analysis} sensitivity", total=len(jobs), unit="model"
        ) as progress:
            for _, inp, estimand in jobs:
                outcome, change_type = inp["outcome"], inp["change_type"]
                job_started = time.perf_counter()
                status = "completed"
                try:
                    frame = _checkpoint_frame(analysis, outcome, change_type, estimand)
                    if frame is not None:
                        status = "checkpoint"
                        checkpoints += 1
                    else:
                        result = run_sensitivity_for_model(
                            inp, estimand, analysis, method=method,
                            timeout_sec=HONESTDID_MODEL_TIMEOUT_SEC,
                        )
                        result["table"].to_csv(
                            _analysis_checkpoint_path(
                                analysis, outcome, change_type, estimand
                            ),
                            index=False,
                        )
                    completed += 1
                except Exception as exc:
                    status = "failed"
                    failed += 1
                    _record_failure(outcome, change_type, estimand, analysis, exc)
                    failure_rows.append({
                        "outcome": outcome, "change_type": change_type,
                        "estimand": estimand, "analysis": analysis,
                        "stage": analysis, "error": repr(exc),
                    })
                finally:
                    elapsed = time.perf_counter() - started
                    remaining = len(jobs) - completed - failed
                    avg = elapsed / max(completed + failed, 1)
                    status_rows.append({
                        "outcome": outcome, "change_type": change_type,
                        "estimand": estimand, "analysis": analysis, "status": status,
                        "elapsed_seconds": time.perf_counter() - job_started,
                    })
                    progress.update(1)
                    progress.set_postfix(
                        outcome=outcome, change_type=change_type, estimand=estimand,
                        analysis=analysis, completed=completed, failed=failed,
                        checkpoint=checkpoints, elapsed=f"{elapsed / 60:.1f}m",
                        eta=f"{avg * remaining / 60:.1f}m", refresh=False,
                    )
                    pd.DataFrame(
                        failure_rows,
                        columns=[
                            "outcome", "change_type", "estimand", "analysis",
                            "stage", "error",
                        ],
                    ).drop_duplicates().to_csv(
                        _analysis_failure_path(analysis), index=False
                    )
                    gc.collect()
    finally:
        results, conclusions = _rebuild_analysis_state(analysis)
        status = pd.DataFrame(status_rows)
        analysis_job_status[analysis] = status
        if analysis == "relmag":
            globals()["relmag_results"] = results
            globals()["relmag_conclusion_df"] = conclusions
        elif analysis == "smoothness":
            globals()["smoothness_results"] = results
            globals()["smoothness_conclusion_df"] = conclusions
        else:
            raise ValueError(f"unknown analysis {analysis!r}")
    _guard_analysis_state(analysis, results, conclusions, status)
    return results, conclusions


relmag_results, relmag_conclusion_df = run_analysis_jobs("relmag", "C-LF")
relmag_results.to_csv(OUT_DIR / "relative_magnitude_sensitivity.csv", index=False)
relmag_conclusion_df.to_csv(OUT_DIR / "relative_magnitude_conclusions.csv", index=False)
display(relmag_conclusion_df)


        outcome          change_type                     estimand analysis  \
0       sch_flg          region_only                    on_impact   relmag   
1       sch_flg          region_only                    short_run   relmag   
2       sch_flg          region_only            main_post_average   relmag   
3       sch_flg          region_only  full_supported_post_average   relmag   
4       sch_flg        workmode_only                    on_impact   relmag   
..          ...                  ...                          ...      ...   
75  t_available  region_and_workmode  full_supported_post_average   relmag   
76  t_available          CORE_pooled                    on_impact   relmag   
77  t_available          CORE_pooled                    short_run   relmag   
78  t_available          CORE_pooled            main_post_average   relmag   
79  t_available          CORE_pooled  full_supported_post_average   relmag   

                          conclusion_code    orig_lb    orig_ub

## 12. Чувствительность по гладкости ($\\Delta^{SD}$)

Границы гладкости вторых разностей через `createSensitivityResults` по исход-специфичной сетке $M$, привязанной к калибровке предпериода `M_pre` (максимальная абсолютная вторая разность предтренда; см. `compute_M_pre`). $M = 0$ соответствует точным параллельным трендам. По умолчанию метод `FLCI`. Как и выше, $M$ — bound гладкости, а не дозовой вес.

In [30]:
smoothness_results, smoothness_conclusion_df = run_analysis_jobs("smoothness", "FLCI")
smoothness_results.to_csv(OUT_DIR / "smoothness_sensitivity.csv", index=False)
smoothness_conclusion_df.to_csv(OUT_DIR / "smoothness_conclusions.csv", index=False)
display(smoothness_conclusion_df)


        outcome          change_type                     estimand    analysis  \
0       sch_flg          region_only                    on_impact  smoothness   
1       sch_flg          region_only                    short_run  smoothness   
2       sch_flg          region_only            main_post_average  smoothness   
3       sch_flg          region_only  full_supported_post_average  smoothness   
4       sch_flg        workmode_only                    on_impact  smoothness   
..          ...                  ...                          ...         ...   
75  t_available  region_and_workmode  full_supported_post_average  smoothness   
76  t_available          CORE_pooled                    on_impact  smoothness   
77  t_available          CORE_pooled                    short_run  smoothness   
78  t_available          CORE_pooled            main_post_average  smoothness   
79  t_available          CORE_pooled  full_supported_post_average  smoothness   

                          c

## 13. порог устойчивости-значения

порог устойчивости — наименьший параметр чувствительности ($\bar{M}$ для relative magnitude, $M$ для smoothness), при котором устойчивое доверительное множество впервые включает ноль. Ищется грубой сеткой с уточнением бисекцией. Прикрепляется только формальный `conclusion_code` (без нарративных утверждений). Если базовое доверительное множество уже включает ноль, порог устойчивости отсутствует (`baseline_inconclusive`); если ноль исключён на всём диапазоне поиска — `robust_at_reported_range` с порог устойчивости за пределами сетки.

In [35]:
from statistics import NormalDist

MAX_WORKERS = 1
BREAKDOWN_MAX_NEW_JOBS = None
BREAKDOWN_TEST_KEY = None
# Пример одного задания; ограничение max-new сохраняется:
# BREAKDOWN_TEST_KEY = {"outcome":"sch_flg","change_type":"workmode_only","estimand":"on_impact","analysis":"relmag"}
assert MAX_WORKERS == 1, "Breakdown execution must remain sequential"

BREAKDOWN_CKPT_DIR = OUT_DIR / "checkpoints" / "breakdown"
BREAKDOWN_CKPT_DIR.mkdir(parents=True, exist_ok=True)
BREAKDOWN_KEY_COLS = ["outcome", "change_type", "estimand", "analysis"]
BREAKDOWN_REQUIRED_COLS = set(BREAKDOWN_KEY_COLS) | {
    "orig_lb", "orig_ub", "breakdown_value", "conclusion_code",
    "breakdown_beyond_grid", "status", "elapsed_seconds",
}
BREAKDOWN_ALLOWED_CODES = {
    "baseline_inconclusive", "sign_sensitive_to_moderate_violations",
    "robust_at_reported_range",
}
BREAKDOWN_FAILURE_PATH = OUT_DIR / "breakdown_failures.csv"
BREAKDOWN_FAILURE_COLS = [
    "timestamp", "outcome", "change_type", "estimand", "analysis", "stage",
    "exception_type", "error", "returncode", "stdout_tail", "stderr_tail",
    "config_path", "workdir", "elapsed_seconds", "rscript_calls",
    "honestdid_evaluations", "honestdid_function_calls",
]


def _breakdown_checkpoint_path(inp: dict, estimand: str, analysis: str) -> Path:
    return BREAKDOWN_CKPT_DIR / (
        f"breakdown__{analysis}__{inp['outcome']}__"
        f"{inp['change_type']}__{estimand}.csv"
    )


def _breakdown_key(inp: dict, estimand: str, analysis: str) -> tuple[str, ...]:
    return (inp["outcome"], inp["change_type"], estimand, analysis)


def _atomic_csv_write(frame: pd.DataFrame, path: Path, validator=None) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    staging_path = path.with_name(f".{path.name}.tmp.{os.getpid()}.{time.time_ns()}")
    try:
        frame.to_csv(staging_path, index=False)
        reread = pd.read_csv(staging_path)
        if validator is not None:
            validator(reread)
        os.replace(staging_path, path)
    finally:
        staging_path.unlink(missing_ok=True)


def _existing_failures() -> list[dict]:
    if not BREAKDOWN_FAILURE_PATH.exists() or BREAKDOWN_FAILURE_PATH.stat().st_size == 0:
        return []
    try:
        frame = pd.read_csv(BREAKDOWN_FAILURE_PATH)
    except Exception:
        return []
    for col in BREAKDOWN_FAILURE_COLS:
        if col not in frame:
            frame[col] = ""
    return frame[BREAKDOWN_FAILURE_COLS].fillna("").to_dict("records")


_breakdown_failures = _existing_failures()


def _save_breakdown_failures() -> None:
    frame = pd.DataFrame(_breakdown_failures, columns=BREAKDOWN_FAILURE_COLS)
    frame = frame.drop_duplicates(
        BREAKDOWN_KEY_COLS
        + ["stage", "exception_type", "error", "config_path"],
        keep="last",
    )
    _atomic_csv_write(frame, BREAKDOWN_FAILURE_PATH)


def _append_breakdown_failure(key: tuple[str, ...], stage: str, error) -> None:
    _breakdown_failures.append({
        "timestamp": datetime.now(timezone.utc).isoformat(),
        **dict(zip(BREAKDOWN_KEY_COLS, key)),
        "stage": getattr(error, "stage", stage) or stage,
        "exception_type": type(error).__name__,
        "error": str(error),
        "returncode": getattr(error, "returncode", ""),
        "stdout_tail": getattr(error, "stdout_tail", ""),
        "stderr_tail": getattr(error, "stderr_tail", ""),
        "config_path": getattr(error, "config_path", ""),
        "workdir": getattr(error, "workdir", ""),
        "elapsed_seconds": getattr(error, "elapsed_seconds", np.nan),
        "rscript_calls": getattr(error, "rscript_calls", 0),
        "honestdid_evaluations": getattr(error, "honestdid_evaluations", 0),
        "honestdid_function_calls": getattr(
            error, "honestdid_function_calls", 0
        ),
    })
    _save_breakdown_failures()


def _validate_breakdown_frame(
    frame: pd.DataFrame, expected_key: tuple[str, ...], expected_filename: str,
) -> None:
    if len(frame) != 1:
        raise ValueError(f"{expected_filename}: expected one row, got {len(frame)}")
    missing = BREAKDOWN_REQUIRED_COLS - set(frame.columns)
    if missing:
        raise ValueError(f"{expected_filename}: missing fields {sorted(missing)}")
    actual = tuple(str(frame.iloc[0][col]) for col in BREAKDOWN_KEY_COLS)
    if actual != expected_key:
        raise ValueError(
            f"{expected_filename}: filename/content key mismatch "
            f"expected={expected_key} actual={actual}"
        )
    row = frame.iloc[0]
    if row["status"] != "completed":
        raise ValueError(f"{expected_filename}: status must equal completed")
    if row["conclusion_code"] not in BREAKDOWN_ALLOWED_CODES:
        raise ValueError(f"{expected_filename}: invalid conclusion_code")
    orig = pd.to_numeric(pd.Series([row["orig_lb"], row["orig_ub"]]), errors="coerce")
    if not np.isfinite(orig.to_numpy(dtype=float)).all():
        raise ValueError(f"{expected_filename}: non-finite original bounds")
    elapsed = pd.to_numeric(pd.Series([row["elapsed_seconds"]]), errors="coerce").iloc[0]
    value = pd.to_numeric(pd.Series([row["breakdown_value"]]), errors="coerce").iloc[0]
    if not np.isfinite(elapsed) or float(elapsed) < 0:
        raise ValueError(f"{expected_filename}: non-finite elapsed value")
    if not np.isfinite(value) and row["conclusion_code"] != "baseline_inconclusive":
        raise ValueError(f"{expected_filename}: non-finite breakdown value")


def _load_breakdown_checkpoint(
    inp: dict, estimand: str, analysis: str, **_ignored,
) -> pd.DataFrame | None:
    """Проверяет и дополняет контрольную точку в памяти без перезаписи."""
    path = _breakdown_checkpoint_path(inp, estimand, analysis)
    key = _breakdown_key(inp, estimand, analysis)
    if not path.exists() or path.stat().st_size == 0:
        return None
    try:
        frame = pd.read_csv(path)
        _validate_breakdown_frame(frame, key, path.name)
    except Exception:
        return None
    enriched = frame.copy()
    defaults = {
        "source": "checkpoint", "rscript_calls": 0,
        "honestdid_evaluations": 0, "honestdid_function_calls": 0,
        "coarse_points_reused": 0, "refinement_points_computed": 0,
    }
    for col, value in defaults.items():
        if col not in enriched:
            enriched[col] = value
    return enriched


def _atomic_breakdown_checkpoint(
    row: dict, inp: dict, estimand: str, analysis: str,
) -> None:
    path = _breakdown_checkpoint_path(inp, estimand, analysis)
    key = _breakdown_key(inp, estimand, analysis)
    _atomic_csv_write(
        pd.DataFrame([row]), path,
        validator=lambda frame: _validate_breakdown_frame(frame, key, path.name),
    )


def _validated_sensitivity(
    frame: pd.DataFrame, expected_key: tuple[str, ...], analysis: str,
) -> pd.DataFrame:
    required = set(BREAKDOWN_KEY_COLS) | {"param_value", "lb", "ub"}
    if frame is None or frame.empty or not required.issubset(frame.columns):
        return pd.DataFrame(columns=list(required))
    current = frame.copy()
    for col, value in zip(BREAKDOWN_KEY_COLS, expected_key):
        current = current.loc[current[col].astype(str) == value]
    if current.empty or not current["analysis"].eq(analysis).all():
        return pd.DataFrame(columns=list(required))
    for col in ("param_value", "lb", "ub"):
        current[col] = pd.to_numeric(current[col], errors="coerce")
    current = current.loc[
        np.isfinite(current[["param_value", "lb", "ub"]]).all(axis=1)
    ]
    return current.sort_values("param_value").drop_duplicates(
        "param_value", keep="last"
    )


def _cached_sensitivity(inp: dict, estimand: str, analysis: str) -> pd.DataFrame:
    """Объединяет источники по полному ключу, оставляя одну строку на значение параметра."""
    key = _breakdown_key(inp, estimand, analysis)
    pieces = []
    memory = globals().get(
        "relmag_results" if analysis == "relmag" else "smoothness_results"
    )
    if isinstance(memory, pd.DataFrame) and not memory.empty:
        valid = _validated_sensitivity(memory, key, analysis)
        if not valid.empty:
            pieces.append(valid)
    result_name = (
        "relative_magnitude_sensitivity.csv"
        if analysis == "relmag" else "smoothness_sensitivity.csv"
    )
    candidates = [
        _analysis_checkpoint_path(analysis, inp["outcome"], inp["change_type"], estimand),
        OUT_DIR / "per_model" / f"{analysis}__{inp['outcome']}__{inp['change_type']}.csv",
        OUT_DIR / inp["outcome"] / inp["change_type"] / result_name,
        OUT_DIR / result_name,
    ]
    seen = set()
    for path in candidates:
        path = Path(path)
        identity = str(path.resolve())
        if identity in seen:
            continue
        seen.add(identity)
        if not path.exists() or path.stat().st_size == 0:
            continue
        try:
            valid = _validated_sensitivity(pd.read_csv(path), key, analysis)
        except Exception:
            continue
        if not valid.empty:
            pieces.append(valid)
    if not pieces:
        return pd.DataFrame(columns=["param_value", "lb", "ub"])
    return (
        pd.concat(pieces, ignore_index=True)
        .sort_values("param_value")
        .drop_duplicates("param_value", keep="first")
        .reset_index(drop=True)
    )


def _original_ci_without_r(
    inp: dict, estimand: str, alpha: float = ALPHA,
) -> tuple[float, float]:
    l_vec = np.asarray(inp["l_vecs"][estimand], dtype=float)
    num_pre = int(inp["num_pre"])
    post = np.asarray(inp["betahat"], dtype=float)[num_pre:]
    sigma = np.asarray(inp["sigma"], dtype=float)[num_pre:, num_pre:]
    estimate = float(l_vec @ post)
    variance = max(float(l_vec @ sigma @ l_vec), 0.0)
    half_width = NormalDist().inv_cdf(1.0 - float(alpha) / 2.0) * np.sqrt(variance)
    return estimate - half_width, estimate + half_width


def _original_ci_from_grid_or_python(
    cached: pd.DataFrame, inp: dict, estimand: str,
) -> tuple[float, float, str]:
    if {"orig_lb", "orig_ub"}.issubset(cached.columns) and not cached.empty:
        lb = pd.to_numeric(cached["orig_lb"], errors="coerce").to_numpy(dtype=float)
        ub = pd.to_numeric(cached["orig_ub"], errors="coerce").to_numpy(dtype=float)
        finite = np.isfinite(lb) & np.isfinite(ub)
        if finite.any():
            lb, ub = lb[finite], ub[finite]
            if np.allclose(lb, lb[0], rtol=1e-6, atol=1e-6) and np.allclose(
                ub, ub[0], rtol=1e-6, atol=1e-6
            ):
                return float(lb[0]), float(ub[0]), "existing_grid"
    lb, ub = _original_ci_without_r(inp, estimand, ALPHA)
    if not np.isfinite([lb, ub]).all():
        raise ValueError("Python original CI is non-finite")
    return float(lb), float(ub), "python_formula"


def _coarse_grid(inp: dict, analysis: str, n_grid: int) -> np.ndarray:
    if analysis == "relmag":
        return np.round(np.linspace(0.25, 5.0, n_grid), 4)
    m_ref = (
        inp["M_pre"] if inp["M_pre"] > 0
        else float(np.sqrt(np.max(np.diag(inp["sigma"]))))
    )
    return np.round(np.linspace(0.0, 3.0 * max(m_ref, 1e-6), n_grid), 6)


def plan_breakdown_job(
    inp: dict, estimand: str, analysis: str, *, n_grid: int = 20,
    n_refine: int = 25, tol: float = 1e-3,
) -> dict:
    """Планирует задание по полному ключу без запуска R."""
    cached = _cached_sensitivity(inp, estimand, analysis)
    orig_lb, orig_ub, orig_source = _original_ci_from_grid_or_python(
        cached, inp, estimand
    )
    values = pd.to_numeric(
        cached.get("param_value", pd.Series(dtype=float)), errors="coerce"
    )
    rows = int(len(cached))
    plan = {
        "outcome": inp["outcome"], "change_type": inp["change_type"],
        "estimand": estimand, "analysis": analysis, "grid_rows": rows,
        "grid_min": float(values.min()) if rows else None,
        "grid_max": float(values.max()) if rows else None,
        "first_include_zero": None, "bracket": None,
        "planned_source": None, "planned_rscript_calls": 0,
        "planned_honestdid_evaluations_bound": 0,
        "orig_lb": orig_lb, "orig_ub": orig_ub, "orig_source": orig_source,
    }
    if ci_includes_zero(orig_lb, orig_ub):
        plan["planned_source"] = "existing_grid_baseline"
        return plan
    if rows == 0:
        plan.update(
            planned_source="r_full_search", planned_rscript_calls=1,
            planned_honestdid_evaluations_bound=int(n_grid + n_refine),
        )
        return plan
    ordered = cached.sort_values("param_value").reset_index(drop=True)
    included = [
        ci_includes_zero(float(row.lb), float(row.ub))
        for row in ordered[["lb", "ub"]].itertuples(index=False)
    ]
    hits = [i for i, flag in enumerate(included) if flag]
    if not hits:
        plan["planned_source"] = "existing_grid_beyond"
        return plan
    first = hits[0]
    first_value = float(ordered.iloc[first]["param_value"])
    lo = float(ordered.iloc[first - 1]["param_value"]) if first > 0 else 0.0
    plan["first_include_zero"] = first_value
    plan["bracket"] = (lo, first_value)
    if first_value - lo <= tol:
        plan["planned_source"] = "existing_grid_exact"
        return plan
    needed = int(np.ceil(np.log2((first_value - lo) / tol)))
    plan.update(
        planned_source="r_refinement", planned_rscript_calls=1,
        planned_honestdid_evaluations_bound=min(int(n_refine), max(0, needed)),
    )
    return plan


def plan_breakdown_key(test_key: dict, **kwargs) -> dict:
    if set(test_key) != set(BREAKDOWN_KEY_COLS):
        raise ValueError("test key must contain exactly the four breakdown key fields")
    inp = validated_inputs.get((test_key["outcome"], test_key["change_type"]))
    if inp is None:
        raise KeyError("validated input not found")
    return plan_breakdown_job(
        inp, test_key["estimand"], test_key["analysis"], **kwargs
    )


def breakdown_value(
    inp: dict, estimand: str, analysis: str, method: str,
    n_grid: int = 20, n_refine: int = 25, tol: float = 1e-3,
) -> dict:
    """Вычисляет порог устойчивости не более чем с одним запуском R-моста."""
    started = time.perf_counter()
    plan = plan_breakdown_job(
        inp, estimand, analysis, n_grid=n_grid, n_refine=n_refine, tol=tol
    )
    source = plan["planned_source"]
    record = {
        "outcome": inp["outcome"], "change_type": inp["change_type"],
        "estimand": estimand, "analysis": analysis,
        "orig_lb": plan["orig_lb"], "orig_ub": plan["orig_ub"],
        "breakdown_value": np.nan, "breakdown_beyond_grid": False,
        "conclusion_code": "computation_failed", "status": "completed",
        "source": source, "rscript_calls": 0, "honestdid_evaluations": 0,
        "honestdid_function_calls": 0,
        "coarse_points_reused": int(plan["grid_rows"]),
        "refinement_points_computed": 0, "elapsed_seconds": 0.0,
    }
    if source == "existing_grid_baseline":
        record.update(breakdown_value=0.0, conclusion_code="baseline_inconclusive")
    elif source == "existing_grid_exact":
        record.update(
            breakdown_value=float(plan["first_include_zero"]),
            conclusion_code="sign_sensitive_to_moderate_violations",
        )
    elif source == "existing_grid_beyond":
        record.update(
            breakdown_value=float(plan["grid_max"]), breakdown_beyond_grid=True,
            conclusion_code="robust_at_reported_range",
        )
    else:
        refine_only = source == "r_refinement"
        bracket = plan["bracket"]
        grid = (
            np.array([], dtype=float)
            if refine_only else _coarse_grid(inp, analysis, n_grid)
        )
        payload = run_honestdid_breakdown(
            analysis, inp["betahat"], inp["sigma"], inp["num_pre"], inp["num_post"],
            l_vec=inp["l_vecs"][estimand], coarse_grid=grid, method=method,
            n_refine=n_refine, tol=tol, alpha=ALPHA, refine_only=refine_only,
            bracket_lo=bracket[0] if bracket else None,
            bracket_hi=bracket[1] if bracket else None,
            coarse_points_reused=int(plan["grid_rows"]),
            tag=(
                f"breakdown__{analysis}__{inp['outcome']}__"
                f"{inp['change_type']}__{estimand}"
            ),
        )
        if not np.allclose(
            [payload["orig_lb"], payload["orig_ub"]],
            [plan["orig_lb"], plan["orig_ub"]], rtol=1e-6, atol=1e-6,
        ):
            metadata = payload.get("_bridge_metadata", {})
            raise HonestDiDBridgeError(
                f"Python/R original CI mismatch: python=({plan['orig_lb']}, "
                f"{plan['orig_ub']}) R=({payload['orig_lb']}, {payload['orig_ub']})",
                stage="post_validation",
                returncode=metadata.get("returncode", 0),
                stdout_tail=metadata.get("stdout_tail", ""),
                stderr_tail=metadata.get("stderr_tail", ""),
                config_path=metadata.get("config_path", ""),
                workdir=metadata.get("workdir", ""),
                elapsed_seconds=metadata.get("bridge_elapsed", np.nan),
                rscript_calls=1,
                honestdid_evaluations=int(payload["honestdid_evaluations"]),
                honestdid_function_calls=int(payload["honestdid_function_calls"]),
            )
        record.update(
            orig_lb=float(payload["orig_lb"]), orig_ub=float(payload["orig_ub"]),
            breakdown_value=float(payload["breakdown_value"]),
            breakdown_beyond_grid=bool(payload["breakdown_beyond_grid"]),
            conclusion_code=payload["conclusion_code"], source=payload["source"],
            rscript_calls=int(payload["rscript_calls"]),
            honestdid_evaluations=int(payload["honestdid_evaluations"]),
            honestdid_function_calls=int(payload["honestdid_function_calls"]),
            coarse_points_reused=int(payload["coarse_points_reused"]),
            refinement_points_computed=int(payload["refinement_points_computed"]),
            elapsed_seconds=float(payload["elapsed_seconds"]),
        )
    if source not in {"r_refinement", "r_full_search"}:
        record["elapsed_seconds"] = time.perf_counter() - started
    if not np.isfinite([
        record["orig_lb"], record["orig_ub"], record["breakdown_value"],
        record["elapsed_seconds"],
    ]).all():
        raise ValueError("Breakdown result contains non-finite required values")
    return record


_breakdown_jobs = [
    (inp, estimand, analysis, method)
    for inp in validated_inputs.values()
    for estimand in ESTIMANDS
    for analysis, method in (("relmag", "C-LF"), ("smoothness", "FLCI"))
]
BREAKDOWN_EXPECTED_TOTAL = len(validated_inputs) * len(ESTIMANDS) * 2
assert len(_breakdown_jobs) == BREAKDOWN_EXPECTED_TOTAL
print("Breakdown expected total:", BREAKDOWN_EXPECTED_TOTAL)

_preloaded = {}
for inp, estimand, analysis, _ in _breakdown_jobs:
    frame = _load_breakdown_checkpoint(inp, estimand, analysis)
    if frame is not None:
        _preloaded[_breakdown_key(inp, estimand, analysis)] = frame


def _rebuild_breakdown_results() -> pd.DataFrame:
    frames = []
    for inp, estimand, analysis, _ in _breakdown_jobs:
        frame = _load_breakdown_checkpoint(inp, estimand, analysis)
        if frame is not None:
            frames.append(frame)
    result = (
        pd.concat(frames, ignore_index=True)
        if frames else pd.DataFrame(columns=sorted(BREAKDOWN_REQUIRED_COLS))
    )
    if not result.empty:
        result = result.drop_duplicates(
            BREAKDOWN_KEY_COLS, keep="last"
        ).reset_index(drop=True)
    _atomic_csv_write(result, OUT_DIR / "breakdown_values.csv")
    return result


breakdown_results = _rebuild_breakdown_results()
_breakdown_status = [
    dict(zip(BREAKDOWN_KEY_COLS, key), status="checkpoint") for key in _preloaded
]
_breakdown_completed = len(_preloaded)
_breakdown_checkpoints = len(_preloaded)
_breakdown_newly_computed = 0
_breakdown_failed = 0
_breakdown_attempted = 0
_new_attempt_seconds = []
_remaining_jobs = [
    job for job in _breakdown_jobs
    if _breakdown_key(job[0], job[1], job[2]) not in _preloaded
]
if BREAKDOWN_TEST_KEY is not None:
    if set(BREAKDOWN_TEST_KEY) != set(BREAKDOWN_KEY_COLS):
        raise ValueError("BREAKDOWN_TEST_KEY must contain exactly four key fields")
    requested = tuple(BREAKDOWN_TEST_KEY[col] for col in BREAKDOWN_KEY_COLS)
    _remaining_jobs = [
        job for job in _remaining_jobs
        if _breakdown_key(job[0], job[1], job[2]) == requested
    ]

if BREAKDOWN_MAX_NEW_JOBS == 0:
    _target_dry_run_key = {
        "outcome": "sch_flg", "change_type": "workmode_only",
        "estimand": "on_impact", "analysis": "relmag",
    }
    try:
        breakdown_target_dry_run = plan_breakdown_key(_target_dry_run_key)
        print("Target breakdown dry-run (no R):", breakdown_target_dry_run)
    except (KeyError, ValueError) as exc:
        breakdown_target_dry_run = {"error": str(exc), **_target_dry_run_key}
        print("Target breakdown dry-run unavailable:", breakdown_target_dry_run)

progress = tqdm(
    total=BREAKDOWN_EXPECTED_TOTAL, initial=_breakdown_checkpoints,
    desc="Breakdown values", unit="analysis", leave=True, dynamic_ncols=True,
    mininterval=0.5,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}{postfix}]",
)
_interrupted = False
try:
    for inp, estimand, analysis, method in _remaining_jobs:
        if (
            BREAKDOWN_MAX_NEW_JOBS is not None
            and _breakdown_attempted >= int(BREAKDOWN_MAX_NEW_JOBS)
        ):
            break
        outcome, change_type = inp["outcome"], inp["change_type"]
        current_job = "/".join((outcome, change_type, estimand, analysis))
        _breakdown_attempted += 1
        job_started = time.perf_counter()
        status, job_source = "failed", "failure"
        job_rscript_calls, job_evaluations, job_function_calls = 0, 0, 0
        try:
            row = breakdown_value(inp, estimand, analysis, method)
            _atomic_breakdown_checkpoint(row, inp, estimand, analysis)
            _breakdown_newly_computed += 1
            _breakdown_completed += 1
            status, job_source = "completed", row["source"]
            job_rscript_calls = int(row["rscript_calls"])
            job_evaluations = int(row["honestdid_evaluations"])
            job_function_calls = int(row["honestdid_function_calls"])
            breakdown_results = _rebuild_breakdown_results()
        except Exception as exc:
            _breakdown_failed += 1
            job_source = getattr(exc, "stage", "failure")
            job_rscript_calls = int(getattr(exc, "rscript_calls", 0) or 0)
            job_evaluations = int(
                getattr(exc, "honestdid_evaluations", 0) or 0
            )
            job_function_calls = int(
                getattr(exc, "honestdid_function_calls", 0) or 0
            )
            _record_failure(
                outcome, change_type, estimand, f"breakdown_{analysis}", exc
            )
            _append_breakdown_failure(
                (outcome, change_type, estimand, analysis),
                f"breakdown_{analysis}", exc,
            )
        finally:
            elapsed = time.perf_counter() - job_started
            _new_attempt_seconds.append(elapsed)
            _breakdown_status.append({
                "outcome": outcome, "change_type": change_type,
                "estimand": estimand, "analysis": analysis, "status": status,
            })
            progress.update(1)
            progress.set_postfix(
                completed=_breakdown_completed, checkpoint=_breakdown_checkpoints,
                newly_computed=_breakdown_newly_computed, failed=_breakdown_failed,
                current_job=current_job, source=job_source,
                job_elapsed=f"{elapsed:.1f}s", rscript_calls=job_rscript_calls,
                honestdid_evaluations=job_evaluations,
                honestdid_function_calls=job_function_calls, refresh=True,
            )
except KeyboardInterrupt:
    _interrupted = True
    breakdown_results = _rebuild_breakdown_results()
    remaining = BREAKDOWN_EXPECTED_TOTAL - len(breakdown_results)
    print(
        "Breakdown interrupted safely. "
        f"completed={len(breakdown_results)} remaining={remaining} "
        f"failed={_breakdown_failed} checkpoint_dir={BREAKDOWN_CKPT_DIR}"
    )
    raise
finally:
    progress.close()
    if not _interrupted:
        breakdown_results = _rebuild_breakdown_results()

breakdown_job_status = pd.DataFrame(_breakdown_status)
_breakdown_remaining = BREAKDOWN_EXPECTED_TOTAL - len(breakdown_results)
_breakdown_avg_new_seconds = (
    float(np.mean(_new_attempt_seconds)) if _new_attempt_seconds else np.nan
)
print(
    f"Breakdown state: completed={len(breakdown_results)} "
    f"remaining={_breakdown_remaining} failed={_breakdown_failed} "
    f"checkpoint={_breakdown_checkpoints} newly_computed={_breakdown_newly_computed} "
    f"avg_new_attempt_seconds={_breakdown_avg_new_seconds} "
    f"checkpoint_dir={BREAKDOWN_CKPT_DIR}"
)
display(breakdown_results)


Breakdown expected total: 160


Breakdown state: completed=160 remaining=0 failed=0 checkpoint=25 newly_computed=135 avg_new_attempt_seconds=15.308635022222141 checkpoint_dir=<PROJECT_ROOT>/…


         outcome  change_type                     estimand    analysis  \
0        sch_flg  region_only                    on_impact      relmag   
1        sch_flg  region_only                    on_impact  smoothness   
2        sch_flg  region_only                    short_run      relmag   
3        sch_flg  region_only                    short_run  smoothness   
4        sch_flg  region_only            main_post_average      relmag   
..           ...          ...                          ...         ...   
155  t_available  CORE_pooled                    short_run  smoothness   
156  t_available  CORE_pooled            main_post_average      relmag   
157  t_available  CORE_pooled            main_post_average  smoothness   
158  t_available  CORE_pooled  full_supported_post_average      relmag   
159  t_available  CORE_pooled  full_supported_post_average  smoothness   

       orig_lb    orig_ub  breakdown_value  breakdown_beyond_grid  \
0   -10.829311  11.127572              NaN

## 14. Результаты для pooled CORE

Чувствительность для объединённого CORE-воздействия (`change_type == "CORE_pooled"`, т.е. `change_type.isin(CORE_CHANGE_TYPES)`) — headline-спецификация, объединяющая три типа CORE-реконфигурации.

In [36]:
with cell_progress("Plot pooled sensitivity", total=2 * len(ANALYSIS_OUTCOMES), unit="plot") as progress:
    _guard_analysis_state("relmag", relmag_results, relmag_conclusion_df, analysis_job_status["relmag"])
    _guard_analysis_state("smoothness", smoothness_results, smoothness_conclusion_df, analysis_job_status["smoothness"])
    def plot_sensitivity(sens: pd.DataFrame, title: str, stem: str, unit: str):
        """Строит устойчивый доверительный интервал по параметру чувствительности с базовой полосой и линией нуля."""
        if sens.empty:
            return None
        sens = sens.sort_values("param_value")
        unit_label = "days" if unit == "days" else "percentage points"
        with plot_style():
            fig, ax = plt.subplots(figsize=(6.4, 3.6))
            ax.axhline(0.0, color=PALETTE["zero"], linewidth=0.8)
            ax.fill_between(sens["param_value"], sens["lb"], sens["ub"], alpha=0.25,
                            color=PALETTE["treated"], label="robust CI")
            ax.plot(sens["param_value"], sens["lb"], color=PALETTE["treated"], linewidth=1.0)
            ax.plot(sens["param_value"], sens["ub"], color=PALETTE["treated"], linewidth=1.0)
            orig_lb = float(sens["orig_lb"].iloc[0])
            orig_ub = float(sens["orig_ub"].iloc[0])
            ax.axhspan(orig_lb, orig_ub, alpha=0.12, color=PALETTE["control"], label="baseline CI")
            ax.set_xlabel(f"{sens['param_name'].iloc[0]} (sensitivity parameter)")
            ax.set_ylabel(f"ATT ({unit_label})")
            ax.set_title(title)
            ax.legend(loc="best")
            fig.tight_layout()
            save_figure(fig, FIG_DIR / f"{stem}.pdf", preview_png=True, preview_dpi=300)
        return stem


    def sensitivity_subset(results: pd.DataFrame, outcome: str, change_type: str,
                           estimand: str) -> pd.DataFrame:
        if results.empty:
            return results
        return results[
            (results["outcome"] == outcome)
            & (results["change_type"] == change_type)
            & (results["estimand"] == estimand)
        ].copy()


    pooled_conclusions = pd.DataFrame()
    if not relmag_conclusion_df.empty:
        pooled_conclusions = relmag_conclusion_df[
            relmag_conclusion_df["change_type"] == POOLED_LABEL
        ].copy()
        if not pooled_conclusions.empty:
            display(pooled_conclusions)
        for _outcome in ANALYSIS_OUTCOMES:
            _sub = sensitivity_subset(relmag_results, _outcome, POOLED_LABEL, "main_post_average")
            if not _sub.empty:
                plot_sensitivity(
                    _sub, f"{_outcome} x CORE_pooled - relative magnitude (main_post_average)",
                    f"{_outcome}_CORE_pooled_main_post_average_relative_magnitude",
                    unit=_sub["unit"].iloc[0],
                )
            progress.update(1)
    print("Pooled CORE conclusion rows:", len(pooled_conclusions))

    # Графики гладкости для объединённой CORE-выборки (main_post_average)
    if not smoothness_results.empty:
        for _outcome in ANALYSIS_OUTCOMES:
            _sub = sensitivity_subset(smoothness_results, _outcome, POOLED_LABEL, "main_post_average")
            if not _sub.empty:
                plot_sensitivity(
                    _sub, f"{_outcome} x CORE_pooled - smoothness (main_post_average)",
                    f"{_outcome}_CORE_pooled_main_post_average_smoothness",
                    unit=_sub["unit"].iloc[0],
                )
            progress.update(1)



           outcome  change_type                     estimand analysis  \
12         sch_flg  CORE_pooled                    on_impact   relmag   
13         sch_flg  CORE_pooled                    short_run   relmag   
14         sch_flg  CORE_pooled            main_post_average   relmag   
15         sch_flg  CORE_pooled  full_supported_post_average   relmag   
28        meet_flg  CORE_pooled                    on_impact   relmag   
29        meet_flg  CORE_pooled                    short_run   relmag   
30        meet_flg  CORE_pooled            main_post_average   relmag   
31        meet_flg  CORE_pooled  full_supported_post_average   relmag   
44     success_flg  CORE_pooled                    on_impact   relmag   
45     success_flg  CORE_pooled                    short_run   relmag   
46     success_flg  CORE_pooled            main_post_average   relmag   
47     success_flg  CORE_pooled  full_supported_post_average   relmag   
60  utlz_within_25  CORE_pooled                    

Pooled CORE conclusion rows: 20


## 15. Результаты по change_type

Выводы по чувствительности для каждого отдельного CORE change type (`region_only`, `workmode_only`, `region_and_workmode`).

In [37]:
with cell_progress("Plot change-type sensitivity", total=2 * len(CORE_ONLY) * len(ANALYSIS_OUTCOMES), unit="plot") as progress:
    _guard_analysis_state("relmag", relmag_results, relmag_conclusion_df, analysis_job_status["relmag"])
    _guard_analysis_state("smoothness", smoothness_results, smoothness_conclusion_df, analysis_job_status["smoothness"])
    by_change_type = pd.DataFrame()
    if not relmag_conclusion_df.empty:
        by_change_type = relmag_conclusion_df[
            relmag_conclusion_df["change_type"].isin(CORE_ONLY)
        ].copy()
        if not by_change_type.empty:
            display(by_change_type.sort_values(["outcome", "change_type", "estimand"]))
        for _ctype in CORE_ONLY:
            for _outcome in ANALYSIS_OUTCOMES:
                _sub = sensitivity_subset(relmag_results, _outcome, _ctype, "main_post_average")
                if not _sub.empty:
                    plot_sensitivity(
                        _sub, f"{_outcome} x {_ctype} - relative magnitude (main_post_average)",
                        f"{_outcome}_{_ctype}_main_post_average_relative_magnitude",
                        unit=_sub["unit"].iloc[0],
                    )
                progress.update(1)
    print("By-change-type conclusion rows:", len(by_change_type))

    # Графики гладкости по change_type (main_post_average)
    if not smoothness_results.empty:
        for _ctype in CORE_ONLY:
            for _outcome in ANALYSIS_OUTCOMES:
                _sub = sensitivity_subset(smoothness_results, _outcome, _ctype, "main_post_average")
                if not _sub.empty:
                    plot_sensitivity(
                        _sub, f"{_outcome} x {_ctype} - smoothness (main_post_average)",
                        f"{_outcome}_{_ctype}_main_post_average_smoothness",
                        unit=_sub["unit"].iloc[0],
                    )
                progress.update(1)



[HTML-таблица сокращена: ~60 строк; см. сохранённый CSV при наличии экспорта]

By-change-type conclusion rows: 60


## 16. Сравнение по исходам

Компактная матрица выводов по исходам, change types и estimands, объединяющая вердикты относительной величины и smoothness с порог устойчивости-значениями.

In [38]:
with cell_progress("Build analysis comparison", total=1) as progress:
    _guard_analysis_state("relmag", relmag_results, relmag_conclusion_df, analysis_job_status["relmag"])
    _guard_analysis_state("smoothness", smoothness_results, smoothness_conclusion_df, analysis_job_status["smoothness"])
    comparison = pd.DataFrame()
    frames = []
    if not relmag_conclusion_df.empty:
        frames.append(relmag_conclusion_df.assign(analysis="relmag"))
    if not smoothness_conclusion_df.empty:
        frames.append(
            smoothness_conclusion_df[
                ["outcome", "change_type", "estimand", "analysis", "conclusion_code",
                 "orig_lb", "orig_ub"]
            ]
        )
    if frames:
        comparison = pd.concat(frames, ignore_index=True)
        if not breakdown_results.empty:
            comparison = comparison.merge(
                breakdown_results[["outcome", "change_type", "estimand", "analysis",
                                   "breakdown_value"]],
                on=["outcome", "change_type", "estimand", "analysis"], how="left",
            )
        comparison_matrix = comparison.pivot_table(
            index=["outcome", "change_type", "estimand"], columns="analysis",
            values="conclusion_code", aggfunc="first",
        )
        comparison.to_csv(OUT_DIR / "cross_outcome_comparison.csv", index=False)
        display(comparison_matrix)
    print("Comparison rows:", len(comparison))

analysis                                                                       relmag  \
outcome        change_type         estimand                                             
meet_flg       CORE_pooled         full_supported_post_average  baseline_inconclusive   
                                   main_post_average            baseline_inconclusive   
                                   on_impact                    baseline_inconclusive   
                                   short_run                    baseline_inconclusive   
               region_and_workmode full_supported_post_average  baseline_inconclusive   
...                                                                               ...   
utlz_within_25 region_only         short_run                    baseline_inconclusive   
               workmode_only       full_supported_post_average  baseline_inconclusive   
                                   main_post_average            baseline_inconclusive   
                     

Comparison rows: 160


## 17. Машиночитаемые экспорты

анализ чувствительности CSV по `outcome`/`change_type`, сводка выводов, реестр моделей (`outcome, change_type, sample_rule, cohort_exclusions, event_window, estimands, status`), блок `session_info`, `run_manifest` и журнал `failed_models` — всё в `outputs/honest_did/`.

In [39]:
import importlib.metadata as ilmd


def _overall_conclusion(relmag_code: str | None, smoothness_code: str | None) -> str:
    required = {relmag_code, smoothness_code}
    if None in required or "computation_failed" in required:
        return "computation_failed"
    if "insufficient_support" in required:
        return "insufficient_support"
    if "baseline_inconclusive" in required:
        return "baseline_inconclusive"
    if "sign_sensitive_to_moderate_violations" in required:
        return "sign_sensitive_to_moderate_violations"
    return "robust_at_reported_range"


def _analysis_conclusion_map(frame: pd.DataFrame, analysis: str) -> dict:
    if frame is None or frame.empty:
        return {}
    current = frame.loc[frame["analysis"] == analysis].drop_duplicates(
        ["outcome", "change_type", "estimand", "analysis"], keep="last"
    )
    return {
        (row["outcome"], row["change_type"], row["estimand"]): row["conclusion_code"]
        for _, row in current.iterrows()
    }


with cell_progress("Export current HonestDiD state", total=1) as progress:
    _guard_analysis_state(
        "relmag", relmag_results, relmag_conclusion_df, analysis_job_status["relmag"]
    )
    _guard_analysis_state(
        "smoothness", smoothness_results, smoothness_conclusion_df,
        analysis_job_status["smoothness"],
    )
    failed_models_df = _dedup_failed_models()

    per_model_dir = OUT_DIR / "per_model"
    per_model_dir.mkdir(parents=True, exist_ok=True)
    for results, label in (
        (relmag_results, "relmag"), (smoothness_results, "smoothness")
    ):
        for (outcome, change_type), group in results.groupby(["outcome", "change_type"]):
            group.to_csv(
                per_model_dir / f"{label}__{outcome}__{change_type}.csv", index=False
            )

    spec_by_outcome = {row["outcome"]: row for row in OUTCOME_SPECS}
    relmag_main = {
        (row["outcome"], row["change_type"]): row["conclusion_code"]
        for _, row in relmag_conclusion_df.loc[
            relmag_conclusion_df["estimand"] == "main_post_average"
        ].iterrows()
    }
    model_registry_rows = []
    for key, info in balanced_windows.items():
        outcome, change_type = key
        spec = spec_by_outcome.get(outcome, {})
        if info["status"] != "ok":
            status = "insufficient_support"
        elif key not in validated_inputs:
            status = "computation_failed"
        else:
            status = relmag_main.get(key, "computation_failed")
        model_registry_rows.append({
            "outcome": outcome, "change_type": change_type,
            "sample_rule": spec.get("sample_rule", ""),
            "cohort_exclusions": (
                f"2022-07-27 {spec.get('cohort_0727', '')}: "
                f"{spec.get('cohort_0727_reason', '')}"
            ),
            "event_window": (
                f"pre={info['pre_weeks']}, ref={REFERENCE_WEEK}, post={info['post_weeks']}"
            ),
            "estimands": ", ".join(ESTIMANDS), "status": status,
        })
    model_registry = pd.DataFrame(model_registry_rows).drop_duplicates(
        ["outcome", "change_type"], keep="last"
    )
    model_registry.to_csv(OUT_DIR / "model_registry.csv", index=False)

    conclusion_summary = pd.concat(
        [relmag_conclusion_df, smoothness_conclusion_df], ignore_index=True
    ).drop_duplicates(SENSITIVITY_KEY_COLS, keep="last")
    conclusion_summary.to_csv(OUT_DIR / "conclusion_summary.csv", index=False)

    def _pkg_version(name: str) -> str:
        try:
            return ilmd.version(name)
        except Exception:
            return "unknown"

    session_info = {
        "python_version": sys.version.split()[0], "platform": platform.platform(),
        "seed": SEED, "numpy": _pkg_version("numpy"), "pandas": _pkg_version("pandas"),
        "matplotlib": _pkg_version("matplotlib"),
        "differences": _pkg_version("differences"), "rscript_path": RSCRIPT_PATH,
        "r_version": R_VERSION, "r_package_status": R_PACKAGE_STATUS,
        "honestdid_ready": HONESTDID_READY, "run_timestamp_utc": RUN_TIMESTAMP,
    }
    (OUT_DIR / "session_info.json").write_text(
        json.dumps(session_info, indent=2), encoding="utf-8"
    )

    run_manifest = {
        "seed": SEED,
        "flags": {
            "RUN_ATTGT": RUN_ATTGT, "FORCE_CLUSTER_BOOTSTRAP": FORCE_CLUSTER_BOOTSTRAP,
            "N_BOOTSTRAP": N_BOOTSTRAP, "ENABLE_EXTRA": ENABLE_EXTRA,
            "EVENT_WEEK_MIN": EVENT_WEEK_MIN, "EVENT_WEEK_MAX": EVENT_WEEK_MAX,
            "REFERENCE_WEEK": REFERENCE_WEEK, "ALPHA": ALPHA,
        },
        "analysis_outcomes": ANALYSIS_OUTCOMES, "change_types": CHANGE_TYPES,
        "estimands": ESTIMANDS,
        "n_models_ok": int((balanced_support_summary["status"] == "ok").sum()),
        "n_validated": len(validated_inputs), "n_failed": len(failed_models_df),
        "csv_outputs": sorted(p.name for p in OUT_DIR.glob("*.csv")),
        "figure_outputs": sorted(p.name for p in FIG_DIR.glob("*.png")),
        "run_timestamp_utc": RUN_TIMESTAMP,
    }
    (OUT_DIR / "run_manifest.json").write_text(
        json.dumps(run_manifest, indent=2), encoding="utf-8"
    )

    for results, filename in (
        (relmag_results, "relative_magnitude_results.csv"),
        (smoothness_results, "smoothness_results.csv"),
    ):
        for (outcome, change_type), group in results.groupby(["outcome", "change_type"]):
            model_output_dir(outcome, change_type)
            group.to_csv(OUT_DIR / outcome / change_type / filename, index=False)
    for (outcome, change_type), group in breakdown_results.groupby(
        ["outcome", "change_type"]
    ):
        group.to_csv(
            OUT_DIR / outcome / change_type / "breakdown_values.csv", index=False
        )

    relmag_codes = _analysis_conclusion_map(relmag_conclusion_df, "relmag")
    smoothness_codes = _analysis_conclusion_map(
        smoothness_conclusion_df, "smoothness"
    )
    summary_rows = []
    for key, inp in validated_inputs.items():
        outcome, change_type = key
        info = balanced_windows[key]
        base = {
            "outcome": outcome, "change_type": change_type,
            "balanced_event_window": (
                f"[{info.get('window_lo')},{info.get('window_hi')}] omit {REFERENCE_WEEK}"
            ),
            "included_cohorts": "; ".join(info["allowed_cohorts"]),
            "n_hex": info.get("n_hex", np.nan), "n_orders": info.get("n_orders", np.nan),
            "pretrend_pvalue": inp.get("pretrend_pvalue", np.nan),
            "smooth_Mpre": inp.get("M_pre", np.nan),
        }
        for estimand in ESTIMANDS:
            row = dict(base)
            row["estimand"] = estimand
            l_vec = inp["l_vecs"][estimand]
            post = inp["betahat"][inp["num_pre"]:]
            post_sigma = inp["sigma"][inp["num_pre"]:, inp["num_pre"]:]
            estimate = float(l_vec @ post)
            se = float(np.sqrt(max(float(l_vec @ post_sigma @ l_vec), 0.0)))
            row.update(
                original_estimate=estimate,
                original_ci_low=estimate - 1.96 * se,
                original_ci_high=estimate + 1.96 * se,
            )
            for target, tag in (
                (1.0, "relative_Mbar_1"), (2.0, "relative_Mbar_2")
            ):
                sub = relmag_results.loc[
                    (relmag_results["outcome"] == outcome)
                    & (relmag_results["change_type"] == change_type)
                    & (relmag_results["estimand"] == estimand)
                    & np.isclose(relmag_results["param_value"].astype(float), target)
                ]
                row[f"{tag}_ci_low"] = float(sub["lb"].iloc[0]) if not sub.empty else np.nan
                row[f"{tag}_ci_high"] = float(sub["ub"].iloc[0]) if not sub.empty else np.nan
            m_pre = float(inp.get("M_pre", np.nan))
            for ratio, tag in (
                (1.0, "smooth_M_over_Mpre_1"), (2.0, "smooth_M_over_Mpre_2")
            ):
                sub = smoothness_results.loc[
                    (smoothness_results["outcome"] == outcome)
                    & (smoothness_results["change_type"] == change_type)
                    & (smoothness_results["estimand"] == estimand)
                ]
                if np.isfinite(m_pre) and m_pre > 0 and not sub.empty:
                    target = ratio * m_pre
                    hit = sub.loc[
                        np.isclose(
                            sub["param_value"].astype(float), target,
                            rtol=1e-3, atol=1e-8,
                        )
                    ]
                else:
                    hit = pd.DataFrame()
                row[f"{tag}_ci_low"] = float(hit["lb"].iloc[0]) if not hit.empty else np.nan
                row[f"{tag}_ci_high"] = float(hit["ub"].iloc[0]) if not hit.empty else np.nan

            bkey = (
                (breakdown_results["outcome"] == outcome)
                & (breakdown_results["change_type"] == change_type)
                & (breakdown_results["estimand"] == estimand)
            )
            rel_break = breakdown_results.loc[
                bkey & (breakdown_results["analysis"] == "relmag")
            ]
            smooth_break = breakdown_results.loc[
                bkey & (breakdown_results["analysis"] == "smoothness")
            ]
            row["breakdown_Mbar"] = (
                float(rel_break["breakdown_value"].iloc[0])
                if not rel_break.empty else np.nan
            )
            row["breakdown_M"] = (
                float(smooth_break["breakdown_value"].iloc[0])
                if not smooth_break.empty else np.nan
            )
            row["breakdown_M_over_Mpre"] = (
                row["breakdown_M"] / m_pre
                if np.isfinite(row["breakdown_M"]) and np.isfinite(m_pre) and m_pre > 0
                else np.nan
            )
            spec = (outcome, change_type, estimand)
            row["relmag_conclusion_code"] = relmag_codes.get(spec, "computation_failed")
            row["smoothness_conclusion_code"] = smoothness_codes.get(
                spec, "computation_failed"
            )
            row["overall_conclusion_code"] = _overall_conclusion(
                relmag_codes.get(spec), smoothness_codes.get(spec)
            )
            row["conclusion_code"] = row["overall_conclusion_code"]
            summary_rows.append(row)

    for _, support_row in balanced_support_summary.loc[
        balanced_support_summary["status"] == "insufficient_support"
    ].iterrows():
        for estimand in ESTIMANDS:
            summary_rows.append({
                "outcome": support_row["outcome"],
                "change_type": support_row["change_type"], "estimand": estimand,
                "balanced_event_window": "",
                "included_cohorts": support_row.get("allowed_cohorts", ""),
                "n_hex": support_row.get("n_hex", np.nan),
                "n_orders": support_row.get("n_orders", np.nan),
                "relmag_conclusion_code": "insufficient_support",
                "smoothness_conclusion_code": "insufficient_support",
                "overall_conclusion_code": "insufficient_support",
                "conclusion_code": "insufficient_support",
            })

    honest_did_summary = pd.DataFrame(summary_rows).drop_duplicates(
        ["outcome", "change_type", "estimand"], keep="last"
    )
    honest_did_summary.to_csv(OUT_DIR / "honest_did_summary.csv", index=False)
    (OUT_DIR / "session_info.txt").write_text(
        "\n".join(f"{key}: {value}" for key, value in session_info.items()) + "\n",
        encoding="utf-8",
    )
    print("Exports written to:", OUT_DIR)
    display(model_registry)


Exports written to: <PROJECT_ROOT>/…


           outcome          change_type  \
0          sch_flg          region_only   
1          sch_flg        workmode_only   
2          sch_flg  region_and_workmode   
3          sch_flg          CORE_pooled   
4         meet_flg          region_only   
5         meet_flg        workmode_only   
6         meet_flg  region_and_workmode   
7         meet_flg          CORE_pooled   
8      success_flg          region_only   
9      success_flg        workmode_only   
10     success_flg  region_and_workmode   
11     success_flg          CORE_pooled   
12  utlz_within_25          region_only   
13  utlz_within_25        workmode_only   
14  utlz_within_25  region_and_workmode   
15  utlz_within_25          CORE_pooled   
16     t_available          region_only   
17     t_available        workmode_only   
18     t_available  region_and_workmode   
19     t_available          CORE_pooled   

                                          sample_rule  \
0   CORE treated vs never-treated; all 

## 18. Интерпретация и ограничения

**Как читать коды выводов (формально, без нарратива):**

- `robust_at_reported_range` — базовое доверительное множество исключает ноль и продолжает исключать ноль на всём отчётном диапазоне $\bar{M}$/$M$. Знак/значимость эффекта не переворачивается нарушениями параллельных трендов до отчётной величины. Это *не* подтверждение гипотезы, а ограниченное утверждение об устойчивости (проверка устойчивости).
- `sign_sensitive_to_moderate_violations` — базовая оценка исключает ноль, но при некотором нарушении параллельных трендов в отчётном диапазоне устойчивое доверительное множество включает ноль; порог устойчивости количественно задаёт, насколько большим должно быть нарушение.
- `baseline_inconclusive` — базовое доверительное множество уже включает ноль; никакой параметр чувствительности не делает неоднозначную оценку однозначной.
- `insufficient_support` — не удалось сформировать сбалансированное окно событийной недели с общими когортами.
- `computation_failed` — сбой оценивания, извлечения ковариации или R-моста; см. `failed_models.csv`.

**Ограничения (повтор предупреждения из начала).** HonestDiD ослабляет только параллельные тренды. Не учитываются anticipation, selection в воздействие, правая цензура медленных исходов (`utlz_flg` без cap и отмечен явно как extra), attrition, ошибки измерения, SUTVA/spillover между hex и слабая идентификация при тонких когортах. Доли на графиках — в процентных пунктах; `t_available` — в днях. Дозовые веса $W_h^+$/$W_h^-$ не связаны с параметрами HonestDiD $M$/$\bar{M}$. Недельные `betahat` читаются из дневной ATTgt event-агрегации при `relative_period == 7 * week`; референсная неделя `-1` (day-level base period) нормирована в ноль и исключена из `betahat`. Все формулировки — осторожные ограниченные утверждения об устойчивости.

## 19. Финальные проверки выполнения

Обязательные assertions для путей, CORE-only change types, референсной недели $-1$, весов `l_vec`, структуры ковариации и машиночитаемых экспортов.

In [40]:
with cell_progress("Run final checks", total=1) as progress:
    assert SEED == 20260712, "seed must be fixed at 20260712"
    assert REFERENCE_WEEK == -1, "reference period must be -1"
    assert set(CHANGE_TYPES) == set(CORE_CHANGE_TYPES) | {POOLED_LABEL}
    assert not ({"opened", "closed", "never_active", "other"} & set(CHANGE_TYPES))
    assert not outcome_spec_table.empty
    assert COHORT_0727 == pd.Timestamp("2022-07-27")
    assert COHORT_0727 not in cohorts_for_outcome("meet_flg")
    assert COHORT_0727 in cohorts_for_outcome("sch_flg")
    assert not balanced_support_summary.empty
    assert set(balanced_support_summary["status"]).issubset(
        {"ok", "insufficient_support"}
    )

    if HONESTDID_READY:
        if not validated_inputs:
            raise RuntimeError(
                "HonestDiD ready state has no validated inputs; "
                f"failures_csv={OUT_DIR / 'failed_models.csv'}"
            )
        expected = len(validated_inputs) * len(ESTIMANDS)
        current_registry = {
            (inp["outcome"], inp["change_type"])
            for inp in validated_inputs.values()
        }
        for analysis, results, conclusions in (
            ("relmag", relmag_results, relmag_conclusion_df),
            ("smoothness", smoothness_results, smoothness_conclusion_df),
        ):
            status = analysis_job_status.get(analysis, pd.DataFrame())
            completed = int(status["status"].isin(["completed", "checkpoint"]).sum())
            failed = int(status["status"].eq("failed").sum())
            status_keys = set(
                map(tuple, status[SENSITIVITY_KEY_COLS].to_numpy())
            ) if not status.empty else set()
            expected_keys = {
                (inp["outcome"], inp["change_type"], estimand, analysis)
                for inp in validated_inputs.values() for estimand in ESTIMANDS
            }
            missing = expected_keys - status_keys
            if completed + failed != expected or missing:
                failures_csv = _analysis_failure_path(analysis)
                raise RuntimeError(
                    f"{analysis} incomplete: expected={expected}, completed={completed}, "
                    f"failed={failed}, missing_keys={sorted(missing)}, "
                    f"failures_csv={failures_csv}"
                )
            if completed < 1:
                raise RuntimeError(
                    f"{analysis} has no successful jobs; "
                    f"failures_csv={_analysis_failure_path(analysis)}"
                )
            if results.empty or conclusions.empty:
                raise RuntimeError(f"{analysis} current results/conclusions are empty")
            if not np.isfinite(results[["lb", "ub"]].to_numpy(dtype=float)).all():
                raise RuntimeError(f"{analysis} successful lb/ub contain non-finite values")
            if results.duplicated(RESULT_KEY_COLS).any():
                raise RuntimeError(f"{analysis} has duplicate sensitivity keys")
            if conclusions.duplicated(SENSITIVITY_KEY_COLS).any():
                raise RuntimeError(f"{analysis} has duplicate conclusion keys")
            result_registry = set(
                map(tuple, results[["outcome", "change_type"]].drop_duplicates().to_numpy())
            )
            if not result_registry.issubset(current_registry):
                raise RuntimeError(
                    f"{analysis} contains stale registry keys: "
                    f"{sorted(result_registry - current_registry)}"
                )

    assert not breakdown_results.duplicated(BREAKDOWN_KEY_COLS).any()
    assert set(breakdown_results["analysis"]).issubset({"relmag", "smoothness"})
    assert set(breakdown_results["conclusion_code"]).issubset(CONCLUSION_CODES)
    assert not model_registry.duplicated(["outcome", "change_type"]).any()
    assert not failed_models_df.duplicated(
        ["outcome", "change_type", "estimand", "stage", "error"]
    ).any()

    required_exports = [
        OUT_DIR / "model_registry.csv", OUT_DIR / "session_info.json",
        OUT_DIR / "run_manifest.json", OUT_DIR / "failed_models.csv",
        OUT_DIR / "session_info.txt", OUT_DIR / "honest_did_summary.csv",
        OUT_DIR / "relative_magnitude_sensitivity.csv",
        OUT_DIR / "smoothness_sensitivity.csv", OUT_DIR / "breakdown_values.csv",
    ]
    for path in required_exports:
        if not path.exists() or path.stat().st_size == 0:
            raise RuntimeError(f"Missing or empty required export: {path}")
        if path.suffix == ".csv":
            pd.read_csv(path)

    summary_check = pd.read_csv(OUT_DIR / "honest_did_summary.csv")
    for code_col in (
        "relmag_conclusion_code", "smoothness_conclusion_code",
        "overall_conclusion_code", "conclusion_code",
    ):
        if not set(summary_check[code_col]).issubset(CONCLUSION_CODES):
            raise RuntimeError(f"Invalid codes in summary column {code_col}")
    matched = summary_check.merge(
        breakdown_results[
            ["outcome", "change_type", "estimand", "analysis", "breakdown_value"]
        ],
        on=["outcome", "change_type", "estimand"], how="inner",
    )
    for _, row in matched.iterrows():
        if (
            row["analysis"] == "relmag"
            and pd.notna(row["breakdown_value"])
            and pd.isna(row["breakdown_Mbar"])
        ):
            raise RuntimeError(f"Missing breakdown_Mbar for {tuple(row[BREAKDOWN_KEY_COLS])}")
        if (
            row["analysis"] == "smoothness"
            and pd.notna(row["breakdown_value"])
            and pd.isna(row["breakdown_M"])
        ):
            raise RuntimeError(f"Missing breakdown_M for {tuple(row[BREAKDOWN_KEY_COLS])}")

    for key, inp in validated_inputs.items():
        sigma = np.asarray(inp["sigma"], dtype=float)
        assert inp["num_pre"] >= 2 and inp["num_post"] >= 1
        assert inp["num_pre"] + inp["num_post"] == len(inp["betahat"])
        assert np.allclose(sigma, sigma.T, atol=1e-8)
        assert np.linalg.eigvalsh(sigma).min() >= -1e-8
        for estimand, l_vec in inp["l_vecs"].items():
            assert np.isclose(np.sum(l_vec), 1.0), f"{estimand} weights invalid for {key}"
            assert len(l_vec) == inp["num_post"]
        if sigma.shape[0] > 1:
            off_diagonal = sigma - np.diag(np.diag(sigma))
            assert np.any(np.abs(off_diagonal) > 0), f"diagonal-only covariance for {key}"

    print("All mandatory current-state checks passed.")


All mandatory current-state checks passed.


## Вывод

Этот ноутбук реализует **анализ чувствительности** HonestDiD (Rambachan & Roth, 2023) к ограниченным нарушениям параллельных трендов для DiD-оценок last-mile CORE. Он **не заменяет** основные спецификации и подтверждённые оценки из `final_empirical_recalculation.ipynb`; дополняет их формальными кодами устойчивости (`conclusion_code`) и порог устойчивости-значениями.

**Ключевые ограничения:** метод ослабляет только параллельные тренды; не устраняет проблемы anticipation, selection, правой цензуры, attrition, измерения, spillover и слабой поддержки по когортам. Неотвержение предтренда не доказывает параллельные тренды. Baseline, уже включающий ноль, остаётся неоднозначным при любых параметрах чувствительности. Дозовые веса $W_h^+$/$W_h^-$ не связаны с $M$/$\bar{M}$.

Итоговые таблицы, графики и `honest_did_summary.csv` в `outputs/honest_did/` предназначены для диагностики и проверки устойчивости, а не для отдельной причинной интерпретации без основной спецификации.